# Main Rerun Mainline


# Main Rerun Stage 1 Runtime Config

这个 notebook 固定用于本次 `main rerun` 的 Stage 1：

- `SPLIT_SCHEME = "primary_holdout"`
- 正式 `primary holdout = 2025-07-15 ~ 2025-10-25`
- `STAGE1_OUTER_N_SPLITS = 5`

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

MAIN_RERUN_STAGE1_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "stage1"
MAIN_RERUN_STAGE1_DIR.mkdir(parents=True, exist_ok=True)

os.environ["STAGE_SPLIT_SCHEME"] = "primary_holdout"
os.environ["STAGE1_OUTER_N_SPLITS"] = "5"
os.environ.setdefault("STAGE1_SEARCH_PROFILE", "smoke")
os.environ["STAGE1_RESULTS_DIR"] = str(MAIN_RERUN_STAGE1_DIR)
os.environ["STAGE1_HOLDOUT_ARTIFACT_STEM"] = "primary_holdout"
os.environ["STAGE1_HOLDOUT_SUMMARY_LABEL"] = "primary_holdout"

print("Project root:", PROJECT_ROOT)
print("Main rerun Stage 1 output dir:", MAIN_RERUN_STAGE1_DIR)
print("STAGE_SPLIT_SCHEME =", os.environ["STAGE_SPLIT_SCHEME"])
print("STAGE1_OUTER_N_SPLITS =", os.environ["STAGE1_OUTER_N_SPLITS"])
print("STAGE1_SEARCH_PROFILE =", os.environ["STAGE1_SEARCH_PROFILE"])

In [ ]:
from pathlib import Path
import json
import os

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl-cache")

import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

DATA_PATH = Path("data/integration/event_stage1_aligned_features_v1.csv")
RESULTS_DIR = Path(os.environ.get("STAGE1_RESULTS_DIR", "stage1/artifacts/v3"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
HOLDOUT_ARTIFACT_STEM = os.environ.get("STAGE1_HOLDOUT_ARTIFACT_STEM", "primary_holdout")
HOLDOUT_SUMMARY_LABEL = os.environ.get("STAGE1_HOLDOUT_SUMMARY_LABEL", HOLDOUT_ARTIFACT_STEM)
HOLDOUT_DISPLAY_TITLE = HOLDOUT_SUMMARY_LABEL.replace("_", " ").title()

RANDOM_STATE = 42
SEARCH_PROFILE = os.environ.get("STAGE1_SEARCH_PROFILE", "smoke")

OUTER_N_SPLITS = int(os.environ.get("STAGE1_OUTER_N_SPLITS", "5"))
OUTER_PURGE_DAYS = 2
OUTER_EMBARGO_DAYS = 5

INNER_N_SPLITS = 3
INNER_PURGE_DAYS = 2
INNER_EMBARGO_DAYS = 2

CALIBRATION_N_SPLITS = 3
CALIBRATION_PURGE_DAYS = 2
CALIBRATION_EMBARGO_DAYS = 2

MIN_INITIAL_TRAIN_DAYS = 180
MIN_INITIAL_TRAIN_POSITIVES = 10
GAP_SPLIT_DAYS = 30
LATER_SEGMENT_MIN_PREFIX_DAYS = 30
MIN_VALID_DAYS_PER_FOLD = 20

TOP_K_FRACS = [0.01, 0.03, 0.05, 0.10]

In [ ]:
SPLIT_SCHEME = os.environ.get("STAGE_SPLIT_SCHEME", "primary_holdout")

TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = os.environ.get("STAGE1_HOLDOUT_SUMMARY_LABEL", "primary_holdout")
OUTSIDE_LABEL = "outside_main_sample"

SPLIT_SCHEMES = {
    "primary_holdout": {
        "train_ranges": [
            ("2017-01-20", "2021-01-08"),
            ("2025-01-20", "2025-07-14"),
        ],
        "holdout_range": ("2025-07-15", "2025-10-25"),
    },
    "first_term_final_year_holdout": {
        "train_ranges": [
            ("2017-01-20", "2019-06-30"),
        ],
        "holdout_range": ("2019-07-01", "2020-02-28"),
    },
}


def describe_split_scheme(split_scheme):
    scheme = SPLIT_SCHEMES[split_scheme]
    return {
        "split_scheme": split_scheme,
        "train_ranges": scheme["train_ranges"],
        "holdout_range": scheme["holdout_range"],
        "train_label": TRAIN_LABEL,
        "holdout_label": HOLDOUT_LABEL,
    }


def build_event_split_masks(event_time_values, split_scheme=SPLIT_SCHEME):
    scheme = SPLIT_SCHEMES[split_scheme]
    event_dates = pd.to_datetime(pd.Series(event_time_values), format="mixed", utc=True).dt.normalize()

    train_mask = pd.Series(False, index=event_dates.index)
    for start_text, end_text in scheme["train_ranges"]:
        start = pd.Timestamp(start_text, tz="UTC")
        end = pd.Timestamp(end_text, tz="UTC")
        train_mask |= (event_dates >= start) & (event_dates <= end)

    holdout_start, holdout_end = scheme["holdout_range"]
    holdout_mask = (
        (event_dates >= pd.Timestamp(holdout_start, tz="UTC"))
        & (event_dates <= pd.Timestamp(holdout_end, tz="UTC"))
    )
    return train_mask, holdout_mask


print("Split scheme:", describe_split_scheme(SPLIT_SCHEME))

In [ ]:
NUMERIC_FEATURES = [
    "finbert_pos",
    "finbert_neu",
    "finbert_neg",
    "candidate_priority_high",
    "keyword_score",
    "in_market_hours",
    "spx_ret_1d",
    "dxy_ret_1d",
    "vix_change_1d",
    "vix_change_5d",
    "real_yield_change_1d",
    "real_yield_change_5d",
    "spx_drawdown_1d",
    "spx_drawdown_5d",
    "vix_ma_5d",
    "gold_spx_corr_20d",
    "trend_tariff_z_60d",
    "trend_inflation_z_60d",
    "trend_war_z_60d",
    "trend_tariff_spike",
    "trend_inflation_spike",
    "trend_war_spike",
]

CATEGORICAL_FEATURES = [
    "term_id",
    "source",
]

LEAKAGE_PRONE_COLUMNS = [
    "follow_up_count_all_48h",
    "follow_up_count_relevant_48h",
    "context_mode",
    "has_evidence_span",
    "review_flag",
    "review_reason",
    "review_flag_yes",
    "rule_label",
    "rule_llm_conflict",
    "evidence_span",
    "rule_text",
    "source_text",
    "key_event_id",
    "audit_tier",
    "confidence",
]

MODEL_NAMES = [
    "logistic_regression",
    "xgboost_classifier",
    "linear_svm",
]

CALIBRATION_METHODS = ["sigmoid", "isotonic"]


def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins[1:-1], right=True)
    ece = 0.0
    for idx in range(n_bins):
        mask = bin_ids == idx
        if not mask.any():
            continue
        observed = y_true[mask].mean()
        predicted = y_prob[mask].mean()
        ece += np.abs(observed - predicted) * mask.mean()
    return float(ece)


def top_k_capture_table(y_true, y_prob, fracs):
    out = []
    total_pos = int(np.sum(y_true))
    ranked = pd.DataFrame({"y_true": y_true, "y_prob": y_prob}).sort_values(
        "y_prob", ascending=False
    )
    for frac in fracs:
        k = max(1, int(np.ceil(len(ranked) * frac)))
        top_slice = ranked.head(k)
        captured = int(top_slice["y_true"].sum())
        out.append(
            {
                "top_k_frac": frac,
                "k": k,
                "captured_positives": captured,
                "capture_rate": captured / total_pos if total_pos else np.nan,
                "precision_at_k": top_slice["y_true"].mean(),
            }
        )
    return pd.DataFrame(out)


def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "pr_auc": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "ece": compute_ece(y_true, y_prob),
        "positive_rate": float(np.mean(y_true)),
        "precision_at_0p5": precision_score(y_true, y_pred, zero_division=0),
        "recall_at_0p5": recall_score(y_true, y_pred, zero_division=0),
        "f1_at_0p5": f1_score(y_true, y_pred, zero_division=0),
    }
    if len(np.unique(y_true)) > 1:
        metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        metrics["roc_auc"] = np.nan
    return metrics


def make_preprocessor(numeric_cols, categorical_cols):
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_cols,
            ),
        ],
        remainder="drop",
    )


def split_day_segments(unique_days, gap_days=30):
    unique_days = pd.Index(sorted(pd.Index(unique_days).unique()))
    if len(unique_days) == 0:
        return []

    segments = []
    start = 0
    diffs = np.diff(unique_days.view("int64")) / (24 * 3600 * 1e9)
    for idx, gap in enumerate(diffs, start=1):
        if gap > gap_days:
            segments.append(unique_days[start:idx])
            start = idx
    segments.append(unique_days[start:])
    return segments


def allocate_splits_to_segments(
    segments,
    day_positive_by_segment,
    n_splits,
    min_initial_train_days,
    purge_days,
    embargo_days,
    min_valid_days,
    min_prefix_days_later_segment,
):
    eligible = []
    for seg_idx, segment in enumerate(segments):
        required_prefix_days = (
            min_initial_train_days if seg_idx == 0 else min_prefix_days_later_segment
        )
        required_days = required_prefix_days + purge_days + min_valid_days
        if len(segment) < required_days:
            continue

        positive_days = int((day_positive_by_segment[seg_idx] > 0).sum())
        if positive_days == 0:
            continue

        capacity = int(
            (len(segment) - required_days) // max(min_valid_days + embargo_days, 1)
        ) + 1
        capacity = min(capacity, positive_days)
        if capacity <= 0:
            continue

        eligible.append(
            {
                "segment_idx": seg_idx,
                "segment_len": len(segment),
                "positive_days": positive_days,
                "capacity": capacity,
            }
        )

    if not eligible:
        raise ValueError("No eligible day segments available for the requested split design.")

    allocation = {item["segment_idx"]: 0 for item in eligible}
    remaining = n_splits

    for item in eligible:
        if remaining <= 0:
            break
        allocation[item["segment_idx"]] += 1
        remaining -= 1

    while remaining > 0:
        best_choice = None
        for item in eligible:
            current = allocation[item["segment_idx"]]
            if current >= item["capacity"]:
                continue

            score = (item["positive_days"] / (current + 1), item["segment_len"])
            if best_choice is None or score > best_choice[0]:
                best_choice = (score, item["segment_idx"])

        if best_choice is None:
            break

        allocation[best_choice[1]] += 1
        remaining -= 1

    if remaining > 0:
        raise ValueError("Segment capacities cannot satisfy the requested number of splits.")

    return allocation


def build_segmented_purged_splits(
    frame,
    date_col,
    label_col,
    n_splits,
    purge_days,
    embargo_days,
    min_initial_train_days,
    min_initial_train_positives,
    gap_days,
    min_prefix_days_later_segment,
    min_valid_days,
):
    work = frame.copy()
    sort_cols = [col for col in ["event_time_utc", "event_id"] if col in work.columns]
    work = work.sort_values(sort_cols).reset_index(drop=True)
    work["_cv_day"] = work[date_col].dt.normalize()

    unique_days = pd.Index(sorted(work["_cv_day"].dropna().unique()))
    segments = split_day_segments(unique_days, gap_days=gap_days)
    if len(segments) == 0:
        raise ValueError("No trading days available to build segmented splits.")

    day_positive_series = work.groupby("_cv_day")[label_col].sum()
    day_positive_by_segment = [
        day_positive_series.reindex(segment, fill_value=0).to_numpy()
        for segment in segments
    ]

    allocation = allocate_splits_to_segments(
        segments=segments,
        day_positive_by_segment=day_positive_by_segment,
        n_splits=n_splits,
        min_initial_train_days=min_initial_train_days,
        purge_days=purge_days,
        embargo_days=embargo_days,
        min_valid_days=min_valid_days,
        min_prefix_days_later_segment=min_prefix_days_later_segment,
    )

    splits = []
    prior_days = []
    prior_positive_total = 0
    fold_id = 1

    for seg_idx, segment in enumerate(segments):
        n_segment_splits = allocation.get(seg_idx, 0)
        day_positive = day_positive_by_segment[seg_idx]

        if n_segment_splits == 0:
            prior_days.extend(list(segment))
            prior_positive_total += int(day_positive.sum())
            continue

        prefix_days_required = (
            min_initial_train_days if seg_idx == 0 else min_prefix_days_later_segment
        )
        remaining_positive_needed = max(min_initial_train_positives - prior_positive_total, 0)
        if remaining_positive_needed > 0:
            cumulative_positive = np.cumsum(day_positive)
            positive_hit = np.where(cumulative_positive >= remaining_positive_needed)[0]
            if len(positive_hit) == 0:
                raise ValueError(
                    f"Segment {seg_idx} cannot satisfy the initial positive-count requirement."
                )
            prefix_days_required = max(prefix_days_required, int(positive_hit[0]) + 1)

        validation_start = prefix_days_required + purge_days
        positive_positions = np.flatnonzero(day_positive > 0)
        positive_positions = positive_positions[positive_positions >= validation_start]
        if len(positive_positions) < n_segment_splits:
            raise ValueError(
                f"Segment {seg_idx} does not have enough positive trading days for {n_segment_splits} folds."
            )

        min_validation_days = max(
            min_valid_days,
            int(np.ceil((len(segment) - validation_start) / max(n_segment_splits * 3, 1))),
        )

        for local_fold_id in range(1, n_segment_splits + 1):
            remaining_folds_after = n_segment_splits - local_fold_id
            remaining_positive_positions = positive_positions[positive_positions >= validation_start]
            if len(remaining_positive_positions) < (remaining_folds_after + 1):
                raise ValueError(
                    f"Segment {seg_idx} cannot keep one positive day for each remaining fold."
                )

            target_positive_count = int(
                np.ceil(len(remaining_positive_positions) / (remaining_folds_after + 1))
            )
            target_positive_count = max(1, target_positive_count)

            val_start_pos = validation_start
            candidate_end_pos = remaining_positive_positions[target_positive_count - 1]
            candidate_end_pos = max(candidate_end_pos, val_start_pos + min_validation_days - 1)

            if remaining_folds_after > 0:
                reserve_first_pos = remaining_positive_positions[-remaining_folds_after]
                max_allowed_end = reserve_first_pos - embargo_days - 1
                candidate_end_pos = min(candidate_end_pos, max_allowed_end)
            else:
                candidate_end_pos = max(candidate_end_pos, remaining_positive_positions[-1])

            val_end_pos = min(candidate_end_pos, len(segment) - 1)
            if val_end_pos < val_start_pos:
                raise ValueError(f"Segment {seg_idx} produced an invalid validation window.")

            train_end_pos = val_start_pos - purge_days - 1
            train_days = pd.Index(list(prior_days) + list(segment[: train_end_pos + 1]))
            valid_days = segment[val_start_pos : val_end_pos + 1]

            train_idx = np.flatnonzero(work["_cv_day"].isin(train_days))
            valid_idx = np.flatnonzero(work["_cv_day"].isin(valid_days))

            train_positive = int(work.loc[train_idx, label_col].sum())
            valid_positive = int(work.loc[valid_idx, label_col].sum())
            if train_positive < min_initial_train_positives:
                raise ValueError(
                    f"Fold {fold_id} training split cannot satisfy the minimum positive count."
                )

            purge_start = segment[train_end_pos + 1] if train_end_pos + 1 < len(segment) else pd.NaT
            purge_end = segment[val_start_pos - 1] if val_start_pos - 1 < len(segment) else pd.NaT
            embargo_start = segment[val_end_pos + 1] if val_end_pos + 1 < len(segment) else pd.NaT
            embargo_end_pos = min(val_end_pos + embargo_days, len(segment) - 1)
            embargo_end = segment[embargo_end_pos] if val_end_pos + 1 < len(segment) else pd.NaT

            splits.append(
                {
                    "fold_id": fold_id,
                    "segment_idx": seg_idx,
                    "train_idx": train_idx,
                    "valid_idx": valid_idx,
                    "segment_start_day": segment.min(),
                    "segment_end_day": segment.max(),
                    "train_start_day": train_days.min(),
                    "train_end_day": train_days.max(),
                    "valid_start_day": valid_days.min(),
                    "valid_end_day": valid_days.max(),
                    "purge_start_day": purge_start,
                    "purge_end_day": purge_end,
                    "embargo_start_day": embargo_start,
                    "embargo_end_day": embargo_end,
                    "train_day_count": len(train_days),
                    "valid_day_count": len(valid_days),
                    "train_rows": len(train_idx),
                    "valid_rows": len(valid_idx),
                    "train_positive": train_positive,
                    "valid_positive": valid_positive,
                }
            )

            fold_id += 1
            validation_start = val_end_pos + 1 + embargo_days

        prior_days.extend(list(segment))
        prior_positive_total += int(day_positive.sum())

    if len(splits) != n_splits:
        raise ValueError(
            f"Expected {n_splits} segmented folds, but built {len(splits)} folds."
        )

    return work.drop(columns="_cv_day"), splits


def build_base_bundle(model_name, numeric_cols, categorical_cols, search_profile, scale_pos_weight):
    preprocessor = make_preprocessor(numeric_cols, categorical_cols)

    if search_profile == "protocol":
        grids = {
            "logistic_regression": {
                "model__penalty": ["l1", "l2"],
                "model__C": [0.01, 0.1, 1],
            },
            "xgboost_classifier": {
                "model__n_estimators": [200, 400],
                "model__max_depth": [2, 3],
                "model__learning_rate": [0.03, 0.08],
                "model__min_child_weight": [1, 5],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
                "model__reg_lambda": [1],
                "model__scale_pos_weight": [scale_pos_weight],
            },
            "linear_svm": {
                "model__C": [0.1, 1, 10],
                "model__loss": ["squared_hinge"],
            },
        }
    else:
        grids = {
            "logistic_regression": {
                "model__penalty": ["l1", "l2"],
                "model__C": [0.01, 0.1],
            },
            "xgboost_classifier": {
                "model__n_estimators": [200],
                "model__max_depth": [2, 3],
                "model__learning_rate": [0.05],
                "model__min_child_weight": [1, 5],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
                "model__reg_lambda": [1],
                "model__scale_pos_weight": [scale_pos_weight],
            },
            "linear_svm": {
                "model__C": [0.1, 1],
                "model__loss": ["squared_hinge"],
            },
        }

    if model_name == "logistic_regression":
        estimator = LogisticRegression(
            solver="liblinear",
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE,
        )
    elif model_name == "xgboost_classifier":
        estimator = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
            verbosity=0,
        )
    elif model_name == "linear_svm":
        estimator = LinearSVC(
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )
    return pipeline, grids[model_name]


def choose_cv_for_subset(
    X_train,
    y_train,
    n_splits,
    purge_days,
    embargo_days,
    min_initial_train_days,
    min_initial_train_positives,
):
    inner_frame = X_train.copy()
    inner_frame["label_retreat"] = y_train
    inner_frame["event_time_utc"] = pd.to_datetime(inner_frame["event_time_utc"], utc=True)
    inner_frame["feature_anchor_date"] = pd.to_datetime(inner_frame["feature_anchor_date"], utc=True)

    try:
        _, inner_splits = build_segmented_purged_splits(
            frame=inner_frame,
            date_col="feature_anchor_date",
            label_col="label_retreat",
            n_splits=n_splits,
            purge_days=purge_days,
            embargo_days=embargo_days,
            min_initial_train_days=min_initial_train_days,
            min_initial_train_positives=min_initial_train_positives,
            gap_days=GAP_SPLIT_DAYS,
            min_prefix_days_later_segment=min(
                LATER_SEGMENT_MIN_PREFIX_DAYS,
                max(min_initial_train_days // 2, 10),
            ),
            min_valid_days=max(10, min(MIN_VALID_DAYS_PER_FOLD, max(n_splits * 5, 10))),
        )
        cv = [(item["train_idx"], item["valid_idx"]) for item in inner_splits]
        strategy = "segmented_purged"
    except ValueError:
        cv = TimeSeriesSplit(n_splits=n_splits)
        strategy = "time_series_fallback"
    return cv, strategy


def run_grid_search(X_train, y_train, model_name, numeric_cols, categorical_cols, search_profile):
    positive_count = int(np.sum(y_train))
    negative_count = int(len(y_train) - positive_count)
    if positive_count == 0:
        raise ValueError("Training split has no positive samples.")

    scale_pos_weight = negative_count / positive_count
    estimator, param_grid = build_base_bundle(
        model_name=model_name,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        search_profile=search_profile,
        scale_pos_weight=scale_pos_weight,
    )

    cv, cv_strategy = choose_cv_for_subset(
        X_train=X_train,
        y_train=y_train,
        n_splits=INNER_N_SPLITS,
        purge_days=INNER_PURGE_DAYS,
        embargo_days=INNER_EMBARGO_DAYS,
        min_initial_train_days=max(90, int(MIN_INITIAL_TRAIN_DAYS * 0.5)),
        min_initial_train_positives=max(4, int(MIN_INITIAL_TRAIN_POSITIVES * 0.4)),
    )

    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="average_precision",
        cv=cv,
        refit=True,
        n_jobs=1,
        verbose=0,
    )
    search.fit(X_train, y_train)
    return search, cv_strategy


def get_probability_variants(
    search,
    model_name,
    X_train,
    y_train,
    X_scored,
):
    variants = {}
    meta = []

    best_base = clone(search.best_estimator_)
    best_base.fit(X_train, y_train)

    if model_name in {"logistic_regression", "xgboost_classifier"}:
        variants["raw"] = best_base.predict_proba(X_scored)[:, 1]
        meta.append({"prob_version": "raw", "calibration_cv_strategy": "not_applicable"})

    calibration_cv, calibration_strategy = choose_cv_for_subset(
        X_train=X_train,
        y_train=y_train,
        n_splits=CALIBRATION_N_SPLITS,
        purge_days=CALIBRATION_PURGE_DAYS,
        embargo_days=CALIBRATION_EMBARGO_DAYS,
        min_initial_train_days=max(90, int(MIN_INITIAL_TRAIN_DAYS * 0.5)),
        min_initial_train_positives=max(4, int(MIN_INITIAL_TRAIN_POSITIVES * 0.4)),
    )

    for method in CALIBRATION_METHODS:
        calibrated = CalibratedClassifierCV(
            estimator=clone(search.best_estimator_),
            method=method,
            cv=calibration_cv,
        )
        calibrated.fit(X_train, y_train)
        variants[method] = calibrated.predict_proba(X_scored)[:, 1]
        meta.append(
            {
                "prob_version": method,
                "calibration_cv_strategy": calibration_strategy,
            }
        )

    return variants, meta


def save_progress_csvs(mapping):
    for path, frame in mapping.items():
        if frame is None:
            continue
        if hasattr(frame, "empty") and frame.empty:
            continue
        frame.to_csv(path, index=False)

In [ ]:
df = pd.read_csv(DATA_PATH)
df["event_time_utc"] = pd.to_datetime(df["event_time_utc"], format="mixed", utc=True)
df["event_time_ny"] = pd.to_datetime(df["event_time_ny"], format="mixed", utc=True, errors="coerce")
df["feature_anchor_date"] = pd.to_datetime(df["feature_anchor_date"], format="mixed", utc=True, errors="coerce")
df["target_trade_date"] = pd.to_datetime(df["target_trade_date"], format="mixed", utc=True, errors="coerce")

df = df.sort_values(["event_time_utc", "event_id"]).reset_index(drop=True)

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")
print("Search profile:", SEARCH_PROFILE)

In [ ]:
train_mask, holdout_mask = build_event_split_masks(
    df["event_time_utc"],
    split_scheme=SPLIT_SCHEME,
)

active_numeric = [col for col in NUMERIC_FEATURES if col in df.columns]
active_categorical = [col for col in CATEGORICAL_FEATURES if col in df.columns]
active_features = active_numeric + active_categorical

constant_cols = [
    col for col in active_features if df.loc[train_mask, col].nunique(dropna=False) <= 1
]
active_numeric = [col for col in active_numeric if col not in constant_cols]
active_categorical = [col for col in active_categorical if col not in constant_cols]
active_features = active_numeric + active_categorical

modeling_cols = [
    "event_id",
    "text_id",
    "event_time_utc",
    "feature_anchor_date",
    "target_trade_date",
    "term_id",
    "source",
    "label_retreat",
] + active_features
modeling_cols = list(dict.fromkeys(modeling_cols))

train_df = df.loc[train_mask, modeling_cols].copy().sort_values(["event_time_utc", "event_id"]).reset_index(drop=True)
holdout_df = df.loc[holdout_mask, modeling_cols].copy().sort_values(["event_time_utc", "event_id"]).reset_index(drop=True)

split_summary = pd.DataFrame(
    [
        {
            "split": "train_pool",
            "n_rows": len(train_df),
            "n_positive": int(train_df["label_retreat"].sum()),
            "positive_rate": train_df["label_retreat"].mean(),
            "start": train_df["event_time_utc"].min(),
            "end": train_df["event_time_utc"].max(),
        },
        {
            "split": HOLDOUT_SUMMARY_LABEL,
            "n_rows": len(holdout_df),
            "n_positive": int(holdout_df["label_retreat"].sum()),
            "positive_rate": holdout_df["label_retreat"].mean(),
            "start": holdout_df["event_time_utc"].min(),
            "end": holdout_df["event_time_utc"].max(),
        },
    ]
)
display(split_summary)
print("Active features:", active_features)
print("Excluded leakage-prone columns:", ", ".join([c for c in LEAKAGE_PRONE_COLUMNS if c in df.columns]))

In [ ]:
train_df_cv, outer_splits = build_segmented_purged_splits(
    frame=train_df,
    date_col="feature_anchor_date",
    label_col="label_retreat",
    n_splits=OUTER_N_SPLITS,
    purge_days=OUTER_PURGE_DAYS,
    embargo_days=OUTER_EMBARGO_DAYS,
    min_initial_train_days=MIN_INITIAL_TRAIN_DAYS,
    min_initial_train_positives=MIN_INITIAL_TRAIN_POSITIVES,
    gap_days=GAP_SPLIT_DAYS,
    min_prefix_days_later_segment=LATER_SEGMENT_MIN_PREFIX_DAYS,
    min_valid_days=MIN_VALID_DAYS_PER_FOLD,
)

outer_split_df = pd.DataFrame(outer_splits).drop(columns=["train_idx", "valid_idx"])
display(outer_split_df)

In [ ]:
outer_metric_rows = []
outer_prediction_rows = []
search_rows = []

for fold in outer_splits:
    fold_id = fold["fold_id"]
    outer_train = train_df_cv.iloc[fold["train_idx"]].copy()
    outer_valid = train_df_cv.iloc[fold["valid_idx"]].copy()

    y_outer_train = outer_train["label_retreat"].astype(int).to_numpy()
    y_outer_valid = outer_valid["label_retreat"].astype(int).to_numpy()

    feature_train = outer_train[["event_time_utc", "feature_anchor_date"] + active_features].copy()
    feature_valid = outer_valid[["event_time_utc", "feature_anchor_date"] + active_features].copy()

    print(
        f"Fold {fold_id}: train_rows={len(outer_train)}, train_pos={int(y_outer_train.sum())}, "
        f"valid_rows={len(outer_valid)}, valid_pos={int(y_outer_valid.sum())}"
    )

    for model_name in MODEL_NAMES:
        print(f"  searching {model_name} ...")
        search, inner_cv_strategy = run_grid_search(
            X_train=feature_train,
            y_train=y_outer_train,
            model_name=model_name,
            numeric_cols=active_numeric,
            categorical_cols=active_categorical,
            search_profile=SEARCH_PROFILE,
        )
        print(
            f"  best {model_name}: inner_pr_auc={float(search.best_score_):.6f}, "
            f"params={json.dumps(search.best_params_, sort_keys=True)}"
        )

        variants, variant_meta = get_probability_variants(
            search=search,
            model_name=model_name,
            X_train=feature_train,
            y_train=y_outer_train,
            X_scored=feature_valid,
        )

        search_rows.append(
            {
                "fold_id": fold_id,
                "model_name": model_name,
                "best_params": json.dumps(search.best_params_, sort_keys=True),
                "inner_best_pr_auc": float(search.best_score_),
                "inner_cv_strategy": inner_cv_strategy,
            }
        )

        meta_lookup = {row["prob_version"]: row for row in variant_meta}
        for prob_version, prob in variants.items():
            metrics = compute_binary_metrics(y_outer_valid, prob, threshold=0.5)
            outer_metric_rows.append(
                {
                    "fold_id": fold_id,
                    "model_name": model_name,
                    "prob_version": prob_version,
                    "inner_best_pr_auc": float(search.best_score_),
                    "inner_cv_strategy": inner_cv_strategy,
                    "calibration_cv_strategy": meta_lookup[prob_version]["calibration_cv_strategy"],
                    "train_rows": len(outer_train),
                    "train_positive": int(y_outer_train.sum()),
                    "valid_rows": len(outer_valid),
                    "valid_positive": int(y_outer_valid.sum()),
                    **metrics,
                }
            )

            pred_df = outer_valid[
                ["event_id", "text_id", "event_time_utc", "feature_anchor_date", "target_trade_date", "label_retreat"]
            ].copy()
            pred_df["fold_id"] = fold_id
            pred_df["model_name"] = model_name
            pred_df["prob_version"] = prob_version
            pred_df["p_retreat"] = prob
            pred_df["is_oof"] = 1
            outer_prediction_rows.append(pred_df)

        save_progress_csvs(
            {
                RESULTS_DIR / "_progress_outer_fold_metrics.csv": pd.DataFrame(outer_metric_rows),
                RESULTS_DIR / "_progress_oof_predictions_all_variants.csv": pd.concat(outer_prediction_rows, ignore_index=True),
                RESULTS_DIR / "_progress_best_params_by_fold.csv": pd.DataFrame(search_rows),
            }
        )

outer_metrics_df = pd.DataFrame(outer_metric_rows)
outer_predictions_df = pd.concat(outer_prediction_rows, ignore_index=True)
search_df = pd.DataFrame(search_rows)

outer_summary_df = (
    outer_metrics_df.groupby(["model_name", "prob_version"])
    .agg(
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
        mean_brier=("brier", "mean"),
        mean_ece=("ece", "mean"),
        mean_roc_auc=("roc_auc", "mean"),
        mean_recall_0p5=("recall_at_0p5", "mean"),
        mean_precision_0p5=("precision_at_0p5", "mean"),
    )
    .reset_index()
    .sort_values(["mean_brier", "mean_ece", "mean_pr_auc"], ascending=[True, True, False])
)

display(outer_metrics_df)
display(outer_summary_df)

In [ ]:
calibrated_selection_pool = outer_summary_df[
    outer_summary_df["prob_version"].isin(CALIBRATION_METHODS)
].copy()
if calibrated_selection_pool.empty:
    raise ValueError("No calibrated variants available for Stage 1 final selection.")

selected_variant_df = (
    calibrated_selection_pool.sort_values(
        ["mean_pr_auc", "mean_brier", "mean_ece"],
        ascending=[False, True, True],
    )
    .head(1)
    .copy()
)
selected_variant_df["variant_key"] = (
    selected_variant_df["model_name"] + "__" + selected_variant_df["prob_version"]
)
selected_variant_df["selection_rule"] = (
    "calibrated_only__pr_auc_first__brier_ece_tiebreak"
)

display(selected_variant_df)
print("Final Stage 1 variant:", selected_variant_df.iloc[0]["variant_key"])

selected_oof_df = outer_predictions_df.merge(
    selected_variant_df[["model_name", "prob_version"]],
    how="inner",
    on=["model_name", "prob_version"],
)

selected_oof_df = selected_oof_df.sort_values(["event_time_utc", "event_id"]).reset_index(drop=True)
display(selected_oof_df.head())

In [ ]:
plt.figure(figsize=(12, 6))
plot_df = outer_summary_df.copy()
plot_df["model_variant"] = plot_df["model_name"] + " | " + plot_df["prob_version"]
sns.barplot(data=plot_df, x="model_variant", y="mean_pr_auc", palette="deep")
plt.title("Outer CV Mean PR-AUC by Model Variant")
plt.xlabel("model_variant")
plt.ylabel("mean PR-AUC")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
holdout_metric_rows = []
holdout_prediction_frames = []
holdout_topk_frames = []

feature_train_full = train_df_cv[["event_time_utc", "feature_anchor_date"] + active_features].copy()
y_train_full = train_df_cv["label_retreat"].astype(int).to_numpy()
feature_holdout = holdout_df[["event_time_utc", "feature_anchor_date"] + active_features].copy()
y_holdout = holdout_df["label_retreat"].astype(int).to_numpy()

for model_name in MODEL_NAMES:
    print(f"{HOLDOUT_SUMMARY_LABEL} refit for {model_name} ...")
    search, inner_cv_strategy = run_grid_search(
        X_train=feature_train_full,
        y_train=y_train_full,
        model_name=model_name,
        numeric_cols=active_numeric,
        categorical_cols=active_categorical,
        search_profile=SEARCH_PROFILE,
    )

    variants, variant_meta = get_probability_variants(
        search=search,
        model_name=model_name,
        X_train=feature_train_full,
        y_train=y_train_full,
        X_scored=feature_holdout,
    )
    meta_lookup = {row["prob_version"]: row for row in variant_meta}

    for prob_version, prob in variants.items():
        metrics = compute_binary_metrics(y_holdout, prob, threshold=0.5)
        holdout_metric_rows.append(
            {
                "model_name": model_name,
                "prob_version": prob_version,
                "best_params": json.dumps(search.best_params_, sort_keys=True),
                "cv_best_pr_auc": float(search.best_score_),
                "inner_cv_strategy": inner_cv_strategy,
                "calibration_cv_strategy": meta_lookup[prob_version]["calibration_cv_strategy"],
                **metrics,
            }
        )

        topk = top_k_capture_table(y_holdout, prob, TOP_K_FRACS)
        topk["model_name"] = model_name
        topk["prob_version"] = prob_version
        holdout_topk_frames.append(topk)

        pred_df = holdout_df[
            ["event_id", "text_id", "event_time_utc", "feature_anchor_date", "target_trade_date", "label_retreat"]
        ].copy()
        pred_df["model_name"] = model_name
        pred_df["prob_version"] = prob_version
        pred_df["p_retreat"] = prob
        pred_df["is_holdout"] = 1
        holdout_prediction_frames.append(pred_df)

    save_progress_csvs(
        {
            RESULTS_DIR / f"_progress_{HOLDOUT_ARTIFACT_STEM}_metrics_all_variants.csv": pd.DataFrame(holdout_metric_rows),
            RESULTS_DIR / f"_progress_{HOLDOUT_ARTIFACT_STEM}_topk_all_variants.csv": pd.concat(holdout_topk_frames, ignore_index=True),
            RESULTS_DIR / f"_progress_{HOLDOUT_ARTIFACT_STEM}_predictions_all_variants.csv": pd.concat(holdout_prediction_frames, ignore_index=True),
        }
    )

holdout_metrics_df = pd.DataFrame(holdout_metric_rows).sort_values(
    ["brier", "ece", "pr_auc"], ascending=[True, True, False]
)
holdout_topk_df = pd.concat(holdout_topk_frames, ignore_index=True)
holdout_predictions_df = pd.concat(holdout_prediction_frames, ignore_index=True)

selected_holdout_df = holdout_predictions_df.merge(
    selected_variant_df[["model_name", "prob_version"]],
    how="inner",
    on=["model_name", "prob_version"],
)

selected_holdout_metrics_df = holdout_metrics_df.merge(
    selected_variant_df[["model_name", "prob_version"]],
    how="inner",
    on=["model_name", "prob_version"],
)

display(holdout_metrics_df)
display(selected_holdout_metrics_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_df = selected_holdout_metrics_df.copy()
plot_df["model_variant"] = plot_df["model_name"] + " | " + plot_df["prob_version"]

sns.barplot(data=plot_df, x="model_variant", y="pr_auc", ax=axes[0], palette="muted")
axes[0].set_title(f"Selected {HOLDOUT_DISPLAY_TITLE} PR-AUC")
axes[0].set_xlabel("model_variant")
axes[0].set_ylabel("PR-AUC")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=plot_df, x="model_variant", y="brier", ax=axes[1], palette="crest")
axes[1].set_title(f"Selected {HOLDOUT_DISPLAY_TITLE} Brier")
axes[1].set_xlabel("model_variant")
axes[1].set_ylabel("Brier")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
calibration_selection_path = RESULTS_DIR / "stage1_v3_selected_variants.csv"
outer_split_path = RESULTS_DIR / "stage1_v3_outer_splits.csv"
outer_metrics_path = RESULTS_DIR / "stage1_v3_outer_fold_metrics.csv"
outer_summary_path = RESULTS_DIR / "stage1_v3_outer_summary_metrics.csv"
outer_predictions_path = RESULTS_DIR / "stage1_v3_oof_predictions_all_variants.csv"
selected_oof_path = RESULTS_DIR / "stage1_v3_oof_predictions_selected.csv"
search_path = RESULTS_DIR / "stage1_v3_best_params_by_fold.csv"
holdout_metrics_path = RESULTS_DIR / f"stage1_v3_{HOLDOUT_ARTIFACT_STEM}_metrics_all_variants.csv"
selected_holdout_metrics_path = RESULTS_DIR / f"stage1_v3_{HOLDOUT_ARTIFACT_STEM}_metrics_selected.csv"
holdout_topk_path = RESULTS_DIR / f"stage1_v3_{HOLDOUT_ARTIFACT_STEM}_topk_all_variants.csv"
holdout_predictions_path = RESULTS_DIR / f"stage1_v3_{HOLDOUT_ARTIFACT_STEM}_predictions_all_variants.csv"
selected_holdout_predictions_path = RESULTS_DIR / f"stage1_v3_{HOLDOUT_ARTIFACT_STEM}_predictions_selected.csv"

selected_variant_df.to_csv(calibration_selection_path, index=False)
outer_split_df.to_csv(outer_split_path, index=False)
outer_metrics_df.to_csv(outer_metrics_path, index=False)
outer_summary_df.to_csv(outer_summary_path, index=False)
outer_predictions_df.to_csv(outer_predictions_path, index=False)
selected_oof_df.to_csv(selected_oof_path, index=False)
search_df.to_csv(search_path, index=False)
holdout_metrics_df.to_csv(holdout_metrics_path, index=False)
selected_holdout_metrics_df.to_csv(selected_holdout_metrics_path, index=False)
holdout_topk_df.to_csv(holdout_topk_path, index=False)
holdout_predictions_df.to_csv(holdout_predictions_path, index=False)
selected_holdout_df.to_csv(selected_holdout_predictions_path, index=False)

print("Saved:")
for path in [
    calibration_selection_path,
    outer_split_path,
    outer_metrics_path,
    outer_summary_path,
    outer_predictions_path,
    selected_oof_path,
    search_path,
    holdout_metrics_path,
    selected_holdout_metrics_path,
    holdout_topk_path,
    holdout_predictions_path,
    selected_holdout_predictions_path,
]:
    print(" -", path)

# Stage 2 Feature Engineering


# Main Rerun Stage 2 Feature Runtime Config

- `SPLIT_SCHEME = "primary_holdout"`
- 正式 `primary holdout = 2025-07-15 ~ 2025-10-25`
- 输入固定读取本次 rerun 的 Stage 1 selected predictions
- 当前 Stage 1 final variant：`linear_svm__sigmoid`

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

MAIN_RERUN_STAGE1_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "stage1"
MAIN_RERUN_STAGE2_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "stage2"
MAIN_RERUN_STAGE2_DIR.mkdir(parents=True, exist_ok=True)

os.environ["STAGE_SPLIT_SCHEME"] = "primary_holdout"
os.environ["STAGE2_HOLDOUT_LABEL"] = "primary_holdout"
os.environ["STAGE1_RESULTS_DIR"] = str(MAIN_RERUN_STAGE1_DIR)
os.environ["STAGE1_OOF_PATH"] = str(MAIN_RERUN_STAGE1_DIR / "stage1_v3_oof_predictions_selected.csv")
os.environ["STAGE1_HOLDOUT_PATH"] = str(
    MAIN_RERUN_STAGE1_DIR / "stage1_v3_primary_holdout_predictions_selected.csv"
)
os.environ["STAGE2_FEATURE_RESULTS_DIR"] = str(MAIN_RERUN_STAGE2_DIR)
os.environ["STAGE2_ACTIVE_VARIANTS"] = "linear_svm__sigmoid"
os.environ["STAGE2_ACTIVE_VARIANT_SPECS_JSON"] = '{"linear_svm__sigmoid": {"model_name": "linear_svm", "prob_version": "sigmoid"}}'
os.environ["STAGE2_EVENT_BRIDGE_PATH"] = str(
    MAIN_RERUN_STAGE2_DIR / "stage2_event_daily_bridge_main_rerun.csv"
)
os.environ["STAGE2_DAILY_LONG_PATH"] = str(
    MAIN_RERUN_STAGE2_DIR / "stage2_daily_features_long_v2_main_rerun.csv"
)
os.environ["STAGE2_DAILY_WIDE_PATH"] = str(
    MAIN_RERUN_STAGE2_DIR / "stage2_daily_features_wide_v2_main_rerun.csv"
)

print("Project root:", PROJECT_ROOT)
print("Main rerun Stage 1 input dir:", MAIN_RERUN_STAGE1_DIR)
print("Main rerun Stage 2 output dir:", MAIN_RERUN_STAGE2_DIR)
print("Stage 1 final variant:", "linear_svm__sigmoid")

Project root: /Users/horace/Coding/ML finance/project
Main rerun Stage 1 input dir: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage1
Main rerun Stage 2 output dir: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2
Stage 1 final variant: linear_svm__sigmoid


In [2]:
from pathlib import Path
import json
import os

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl-cache")

import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

ROOT = Path.cwd()
STAGE1_DIR = Path(os.environ.get("STAGE1_RESULTS_DIR", ROOT / "stage1" / "artifacts" / "v3"))
INTEGRATION_DIR = ROOT / "data" / "integration"
RESULTS_DIR = Path(os.environ.get("STAGE2_FEATURE_RESULTS_DIR", ROOT / "stage2" / "artifacts"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OOF_PATH = Path(os.environ.get("STAGE1_OOF_PATH", STAGE1_DIR / "stage1_v3_oof_predictions_selected.csv"))
HOLDOUT_PATH = Path(
    os.environ.get(
        "STAGE1_HOLDOUT_PATH",
        STAGE1_DIR / "stage1_v3_primary_holdout_predictions_selected.csv",
    )
)
MARKET_PATH = INTEGRATION_DIR / "market_macro_features_daily.csv"
TRENDS_PATH = INTEGRATION_DIR / "trends_features_daily.csv"
BRIDGE_PATH = Path(
    os.environ.get(
        "STAGE2_EVENT_BRIDGE_PATH",
        str(RESULTS_DIR / "stage2_event_daily_bridge_v1.csv"),
    )
)
LONG_PATH = Path(
    os.environ.get(
        "STAGE2_DAILY_LONG_PATH",
        str(RESULTS_DIR / "stage2_daily_features_long_v2.csv"),
    )
)
WIDE_PATH = Path(
    os.environ.get(
        "STAGE2_DAILY_WIDE_PATH",
        str(RESULTS_DIR / "stage2_daily_features_wide_v2.csv"),
    )
)

VARIANT_SPECS = {
    "svm_sigmoid": {
        "model_name": "linear_svm",
        "prob_version": "sigmoid",
    },
    "xgb_sigmoid": {
        "model_name": "xgboost_classifier",
        "prob_version": "sigmoid",
    },
}
runtime_variant_specs_text = os.environ.get("STAGE2_ACTIVE_VARIANT_SPECS_JSON", "").strip()
if runtime_variant_specs_text:
    runtime_variant_specs = json.loads(runtime_variant_specs_text)
    if not isinstance(runtime_variant_specs, dict) or not runtime_variant_specs:
        raise ValueError("STAGE2_ACTIVE_VARIANT_SPECS_JSON must be a non-empty JSON object.")
    VARIANT_SPECS = runtime_variant_specs

active_variant_default = ",".join(VARIANT_SPECS.keys())
ACTIVE_VARIANT_KEYS = [
    key.strip()
    for key in os.environ.get("STAGE2_ACTIVE_VARIANTS", active_variant_default).split(",")
    if key.strip()
]
unknown_variant_keys = [key for key in ACTIVE_VARIANT_KEYS if key not in VARIANT_SPECS]
if unknown_variant_keys:
    raise ValueError(f"Unknown STAGE2_ACTIVE_VARIANTS entries: {unknown_variant_keys}")
ACTIVE_VARIANTS = {key: VARIANT_SPECS[key] for key in ACTIVE_VARIANT_KEYS}

HIGH_P_TACO_THRESHOLD = 0.10
TAIL_THRESHOLD_005 = 0.05
TAIL_THRESHOLD_010 = 0.10

In [3]:
SPLIT_SCHEME = os.environ.get("STAGE_SPLIT_SCHEME", "primary_holdout")

TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = os.environ.get("STAGE2_HOLDOUT_LABEL", "primary_holdout")
OUTSIDE_LABEL = "outside_main_sample"

SPLIT_SCHEMES = {
    "primary_holdout": {
        "train_ranges": [
            ("2017-01-20", "2021-01-08"),
            ("2025-01-20", "2025-07-14"),
        ],
        "holdout_range": ("2025-07-15", "2025-10-25"),
    },
    "first_term_final_year_holdout": {
        "train_ranges": [
            ("2017-01-20", "2019-06-30"),
        ],
        "holdout_range": ("2019-07-01", "2020-02-28"),
    },
}


def describe_split_scheme(split_scheme):
    scheme = SPLIT_SCHEMES[split_scheme]
    return {
        "split_scheme": split_scheme,
        "train_ranges": scheme["train_ranges"],
        "holdout_range": scheme["holdout_range"],
        "train_label": TRAIN_LABEL,
        "holdout_label": HOLDOUT_LABEL,
    }


def classify_daily_split(date_values, split_scheme=SPLIT_SCHEME):
    scheme = SPLIT_SCHEMES[split_scheme]
    dates = pd.to_datetime(pd.Series(date_values)).dt.normalize()
    tz = getattr(dates.dt, "tz", None)
    if tz is not None:
        dates = dates.dt.tz_convert(None)

    train_mask = pd.Series(False, index=dates.index)
    for start_text, end_text in scheme["train_ranges"]:
        start = pd.Timestamp(start_text)
        end = pd.Timestamp(end_text)
        train_mask |= (dates >= start) & (dates <= end)

    holdout_start, holdout_end = scheme["holdout_range"]
    holdout_mask = (dates >= pd.Timestamp(holdout_start)) & (dates <= pd.Timestamp(holdout_end))
    return np.select(
        [train_mask, holdout_mask],
        [TRAIN_LABEL, HOLDOUT_LABEL],
        default=OUTSIDE_LABEL,
    )


print("Split scheme:", describe_split_scheme(SPLIT_SCHEME))

Split scheme: {'split_scheme': 'primary_holdout', 'train_ranges': [('2017-01-20', '2021-01-08'), ('2025-01-20', '2025-07-14')], 'holdout_range': ('2025-07-15', '2025-10-25'), 'train_label': 'train_pool', 'holdout_label': 'primary_holdout'}


## Aggregation V2 Design

在保留 `v1` 的 `max / sum / mean / event_count` 之外，这版新增 4 类 `P_taco` 日聚合：
- 强信号摘要：`p_taco_top1`、`p_taco_top2_sum`
- “至少有一个强信号”的 noisy-or：`p_taco_any = 1 - Π(1 - p_i)`
- 阈值 tail：`p_taco_tail_005`、`p_taco_tail_010`
- 跨日衰减与 stress interaction：如 `p_taco_any_decay_2d`、`p_taco_any_x_vix_ma_5d_lag1`

目标不是机械累加所有小概率事件，而是把 event-level 概率变成更适合 `Stage 2` 的 daily trading signal。

In [4]:
def classify_stage2_date(date_series):
    return classify_daily_split(
        date_series,
        split_scheme=SPLIT_SCHEME,
    )


def load_stage1_predictions():
    oof = pd.read_csv(OOF_PATH)
    holdout = pd.read_csv(HOLDOUT_PATH)

    oof["prediction_source"] = "oof"
    holdout["prediction_source"] = "holdout"
    if "fold_id" not in holdout.columns:
        holdout["fold_id"] = np.nan

    stage1 = pd.concat([oof, holdout], ignore_index=True, sort=False)
    stage1["event_time_utc"] = pd.to_datetime(stage1["event_time_utc"], format="mixed", utc=True)
    stage1["feature_anchor_date"] = pd.to_datetime(stage1["feature_anchor_date"], format="mixed", utc=True)
    stage1["target_trade_date"] = pd.to_datetime(stage1["target_trade_date"], format="mixed", utc=True)
    stage1["target_trade_date_naive"] = stage1["target_trade_date"].dt.tz_convert(None).dt.normalize()

    keep_frames = []
    for variant_key, spec in ACTIVE_VARIANTS.items():
        subset = stage1[
            (stage1["model_name"] == spec["model_name"])
            & (stage1["prob_version"] == spec["prob_version"])
        ].copy()
        subset["variant_key"] = variant_key
        keep_frames.append(subset)

    if not keep_frames:
        raise ValueError("No Stage 1 predictions matched the active Stage 2 variant specs.")
    selected = pd.concat(keep_frames, ignore_index=True)
    return selected


def aggregate_event_probabilities(events):
    grouped_rows = []
    group_keys = ["variant_key", "target_trade_date_naive"]
    for (variant_key, target_date), group in events.groupby(group_keys):
        prob = group["p_retreat"].astype(float).clip(0.0, 1.0)
        grouped_rows.append(
            {
                "variant_key": variant_key,
                "date": target_date,
                "max_p_taco": float(prob.max()),
                "sum_p_taco": float(prob.sum()),
                "mean_p_taco": float(prob.mean()),
                "event_count": int(len(group)),
                "high_p_taco_count": int((prob >= HIGH_P_TACO_THRESHOLD).sum()),
                "p_taco_top1": float(prob.max()),
                "p_taco_top2_sum": float(prob.nlargest(min(2, len(prob))).sum()),
                "p_taco_any": float(1.0 - np.prod(1.0 - prob.to_numpy())),
                "p_taco_tail_005": float(np.maximum(prob - TAIL_THRESHOLD_005, 0.0).sum()),
                "p_taco_tail_010": float(np.maximum(prob - TAIL_THRESHOLD_010, 0.0).sum()),
                "event_count_ge_005": int((prob >= TAIL_THRESHOLD_005).sum()),
                "event_count_ge_010": int((prob >= TAIL_THRESHOLD_010).sum()),
                "event_count_ge_020": int((prob >= 0.20).sum()),
                "source_positive_count": int(group["label_retreat"].sum()),
                "min_event_time_utc": group["event_time_utc"].min(),
                "max_event_time_utc": group["event_time_utc"].max(),
                "prediction_source_first": group["prediction_source"].iloc[0],
            }
        )
    grouped = pd.DataFrame(grouped_rows).sort_values(["variant_key", "date"]).reset_index(drop=True)
    return grouped


def build_daily_base():
    market = pd.read_csv(MARKET_PATH)
    trends = pd.read_csv(TRENDS_PATH)

    market["date"] = pd.to_datetime(market["date"]).dt.normalize()
    trends["date"] = pd.to_datetime(trends["date"]).dt.normalize()

    daily = market.merge(trends, on="date", how="left", validate="one_to_one")
    daily = daily.sort_values("date").reset_index(drop=True)
    daily["stage2_split"] = classify_stage2_date(daily["date"])
    daily["is_main_stage2_sample"] = daily["stage2_split"].isin([TRAIN_LABEL, HOLDOUT_LABEL]).astype(int)
    return daily


def add_lagged_controls(daily):
    lag_cols = [
        "dxy_ret_1d",
        "real_yield_change_1d",
        "real_yield_change_5d",
        "vix_ma_5d",
        "vix_change_5d",
        "spx_ret_1d",
        "spx_drawdown_5d",
        "gold_spx_corr_20d",
        "trend_tariff_z_60d",
        "trend_inflation_z_60d",
        "trend_war_z_60d",
        "trend_tariff_spike",
        "trend_inflation_spike",
        "trend_war_spike",
    ]
    out = daily.copy()
    for col in lag_cols:
        out[f"{col}_lag1"] = out[col].shift(1)
    return out


def build_stage2_long_table(daily_base, aggregated_events):
    long_frames = []
    for variant_key in ACTIVE_VARIANTS:
        taco = aggregated_events[aggregated_events["variant_key"] == variant_key].copy()
        merged = daily_base.merge(
            taco.drop(columns=["variant_key"]),
            on="date",
            how="left",
            validate="one_to_one",
        )
        merged["variant_key"] = variant_key
        merged["max_p_taco"] = merged["max_p_taco"].fillna(0.0)
        merged["sum_p_taco"] = merged["sum_p_taco"].fillna(0.0)
        merged["mean_p_taco"] = merged["mean_p_taco"].fillna(0.0)
        merged["event_count"] = merged["event_count"].fillna(0).astype(int)
        merged["high_p_taco_count"] = merged["high_p_taco_count"].fillna(0).astype(int)
        merged["p_taco_top1"] = merged["p_taco_top1"].fillna(0.0)
        merged["p_taco_top2_sum"] = merged["p_taco_top2_sum"].fillna(0.0)
        merged["p_taco_any"] = merged["p_taco_any"].fillna(0.0)
        merged["p_taco_tail_005"] = merged["p_taco_tail_005"].fillna(0.0)
        merged["p_taco_tail_010"] = merged["p_taco_tail_010"].fillna(0.0)
        merged["event_count_ge_005"] = merged["event_count_ge_005"].fillna(0).astype(int)
        merged["event_count_ge_010"] = merged["event_count_ge_010"].fillna(0).astype(int)
        merged["event_count_ge_020"] = merged["event_count_ge_020"].fillna(0).astype(int)
        merged["source_positive_count"] = merged["source_positive_count"].fillna(0).astype(int)
        merged["has_stage1_signal"] = (merged["event_count"] > 0).astype(int)
        merged["holdout_spillover_flag"] = (
            (merged["prediction_source_first"] == "holdout")
            & (merged["stage2_split"] == "outside_main_sample")
        ).astype(int)
        long_frames.append(merged)
    long_df = pd.concat(long_frames, ignore_index=True)
    return long_df


def add_temporal_taco_features(long_df):
    signal_cols = [
        "p_taco_any",
        "p_taco_top1",
        "p_taco_top2_sum",
        "p_taco_tail_005",
        "p_taco_tail_010",
    ]
    out_frames = []
    for variant_key, frame in long_df.groupby("variant_key", sort=False):
        ordered = frame.sort_values("date").copy()
        for col in signal_cols:
            ordered[f"{col}_decay_2d"] = (
                ordered[col]
                + 0.5 * ordered[col].shift(1).fillna(0.0)
            )
            ordered[f"{col}_decay_3d"] = (
                ordered[col]
                + 0.5 * ordered[col].shift(1).fillna(0.0)
                + 0.25 * ordered[col].shift(2).fillna(0.0)
            )

        ordered["p_taco_any_x_vix_ma_5d_lag1"] = ordered["p_taco_any"] * ordered["vix_ma_5d_lag1"].fillna(0.0)
        ordered["p_taco_any_x_spx_drawdown_5d_lag1"] = (
            ordered["p_taco_any"] * ordered["spx_drawdown_5d_lag1"].fillna(0.0)
        )
        ordered["p_taco_any_x_dxy_ret_1d_lag1"] = (
            ordered["p_taco_any"] * ordered["dxy_ret_1d_lag1"].fillna(0.0)
        )
        out_frames.append(ordered)
    return pd.concat(out_frames, ignore_index=True)


def build_stage2_wide_table(long_df):
    id_cols = [
        "date",
        "stage2_split",
        "is_main_stage2_sample",
        "gold_close",
        "gold_ret_1d",
        "spx_close",
        "vix_close",
        "dxy_close",
        "real_yield_10y",
    ]
    lag_cols = [col for col in long_df.columns if col.endswith("_lag1")]
    keep_taco_cols = [
        col
        for col in long_df.columns
        if (
            col.startswith("p_taco_")
            or col.startswith("event_count")
            or col in ["max_p_taco", "sum_p_taco", "mean_p_taco", "high_p_taco_count", "has_stage1_signal", "holdout_spillover_flag"]
        )
    ]

    wide_base = (
        long_df[id_cols + lag_cols]
        .drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    taco_wide = (
        long_df[["date", "variant_key"] + keep_taco_cols]
        .set_index(["date", "variant_key"])
        .unstack("variant_key")
    )
    taco_wide.columns = [
        f"{variant}__{feature}"
        for feature, variant in taco_wide.columns.to_flat_index()
    ]
    taco_wide = taco_wide.reset_index()

    wide = wide_base.merge(taco_wide, on="date", how="left", validate="one_to_one")
    return wide

In [5]:
stage1_events = load_stage1_predictions()
aggregated_events = aggregate_event_probabilities(stage1_events)
daily_base = build_daily_base()
daily_base_lagged = add_lagged_controls(daily_base)
stage2_long = build_stage2_long_table(daily_base_lagged, aggregated_events)
stage2_long = add_temporal_taco_features(stage2_long)
stage2_wide = build_stage2_wide_table(stage2_long)

print("stage1 selected event rows:", stage1_events.shape)
print("aggregated event rows:", aggregated_events.shape)
print("daily base rows:", daily_base.shape)
print("stage2_long rows:", stage2_long.shape)
print("stage2_wide rows:", stage2_wide.shape)

stage1 selected event rows: (772, 15)
aggregated event rows: (248, 19)
daily base rows: (2293, 38)
stage2_long rows: (2293, 85)
stage2_wide rows: (2293, 54)


In [6]:
event_summary = (
    stage1_events.groupby(["variant_key", "prediction_source"])["label_retreat"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "n_events", "sum": "n_positive", "mean": "positive_rate"})
)
display(event_summary)

aggregated_summary = (
    aggregated_events.groupby("variant_key")[
        [
            "max_p_taco",
            "sum_p_taco",
            "mean_p_taco",
            "event_count",
            "high_p_taco_count",
            "p_taco_any",
            "p_taco_top2_sum",
            "p_taco_tail_010",
            "event_count_ge_010",
        ]
    ]
    .agg(["mean", "max"])
)
display(aggregated_summary)

split_summary = (
    stage2_long.groupby(["variant_key", "stage2_split"])
    .agg(
        n_rows=("date", "count"),
        n_signal_days=("has_stage1_signal", "sum"),
        avg_event_count=("event_count", "mean"),
        avg_max_p_taco=("max_p_taco", "mean"),
    )
)
display(split_summary)

n_events  n_positive  positive_rate
variant_key         prediction_source                                     
linear_svm__sigmoid holdout                 125           6       0.048000
                    oof                     647          31       0.047913

max_p_taco           sum_p_taco           mean_p_taco  \
                          mean       max       mean       max        mean   
variant_key                                                                 
linear_svm__sigmoid   0.080395  0.423721   0.201116  1.909745     0.07098   

                              event_count     high_p_taco_count      \
                          max        mean max              mean max   
variant_key                                                           
linear_svm__sigmoid  0.423721    3.112903  23          0.366935   9   

                    p_taco_any           p_taco_top2_sum            \
                          mean       max            mean       max   
variant_key                                                          
linear_svm__sigmoid   0.172904  0.885447        0.128397  0.654936   

                    p_taco_tail_010           event_count_ge_010      
                               mean       max               mean max  
variant_key                                                           
linear_svm__sigmoid        0.032988  1.009745           0.366935   9

n_rows  n_signal_days  \
variant_key         stage2_split                                 
linear_svm__sigmoid outside_main_sample    1089              0   
                    primary_holdout          74             53   
                    train_pool             1130            194   

                                         avg_event_count  avg_max_p_taco  
variant_key         stage2_split                                          
linear_svm__sigmoid outside_main_sample         0.000000        0.000000  
                    primary_holdout             1.675676        0.048288  
                    train_pool                  0.572566        0.014418

In [7]:
spillover_rows = stage2_long[stage2_long["holdout_spillover_flag"] == 1][
    ["date", "variant_key", "event_count", "max_p_taco", "prediction_source_first", "stage2_split"]
].drop_duplicates()
print("spillover rows outside main sample:", len(spillover_rows))
if len(spillover_rows):
    display(spillover_rows)

spillover rows outside main sample: 0


In [8]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

plot_long = stage2_long[stage2_long["is_main_stage2_sample"] == 1].copy()
sns.lineplot(
    data=plot_long,
    x="date",
    y="max_p_taco",
    hue="variant_key",
    ax=axes[0],
    linewidth=1.8,
)
axes[0].set_title("Daily max_p_taco by variant")
axes[0].set_xlabel("")
axes[0].set_ylabel("max_p_taco")

sns.lineplot(
    data=plot_long,
    x="date",
    y="event_count",
    hue="variant_key",
    ax=axes[1],
    linewidth=1.5,
)
axes[1].set_title("Daily event_count by variant")
axes[1].set_xlabel("date")
axes[1].set_ylabel("event_count")

plt.tight_layout()
plt.show()

In [9]:
keep_long_cols = [
    "date",
    "stage2_split",
    "is_main_stage2_sample",
    "variant_key",
    "gold_close",
    "gold_ret_1d",
    "max_p_taco",
    "sum_p_taco",
    "mean_p_taco",
    "event_count",
    "high_p_taco_count",
    "p_taco_top1",
    "p_taco_top2_sum",
    "p_taco_any",
    "p_taco_tail_005",
    "p_taco_tail_010",
    "event_count_ge_005",
    "event_count_ge_010",
    "event_count_ge_020",
    "p_taco_any_decay_2d",
    "p_taco_any_decay_3d",
    "p_taco_top1_decay_2d",
    "p_taco_top1_decay_3d",
    "p_taco_top2_sum_decay_2d",
    "p_taco_top2_sum_decay_3d",
    "p_taco_tail_005_decay_2d",
    "p_taco_tail_005_decay_3d",
    "p_taco_tail_010_decay_2d",
    "p_taco_tail_010_decay_3d",
    "p_taco_any_x_vix_ma_5d_lag1",
    "p_taco_any_x_spx_drawdown_5d_lag1",
    "p_taco_any_x_dxy_ret_1d_lag1",
    "has_stage1_signal",
    "holdout_spillover_flag",
    "dxy_ret_1d_lag1",
    "real_yield_change_1d_lag1",
    "real_yield_change_5d_lag1",
    "vix_ma_5d_lag1",
    "vix_change_5d_lag1",
    "spx_ret_1d_lag1",
    "spx_drawdown_5d_lag1",
    "gold_spx_corr_20d_lag1",
    "trend_tariff_z_60d_lag1",
    "trend_inflation_z_60d_lag1",
    "trend_war_z_60d_lag1",
    "trend_tariff_spike_lag1",
    "trend_inflation_spike_lag1",
    "trend_war_spike_lag1",
    "source_positive_count",
    "prediction_source_first",
    "min_event_time_utc",
    "max_event_time_utc",
]
stage2_long_export = stage2_long[keep_long_cols].copy()

stage1_events.to_csv(BRIDGE_PATH, index=False)
stage2_long_export.to_csv(LONG_PATH, index=False)
stage2_wide.to_csv(WIDE_PATH, index=False)

print("Saved:")
print(" -", BRIDGE_PATH)
print(" -", LONG_PATH)
print(" -", WIDE_PATH)

Saved:
 - /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/stage2_event_daily_bridge_main_rerun.csv
 - /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/stage2_daily_features_long_v2_main_rerun.csv
 - /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/stage2_daily_features_wide_v2_main_rerun.csv


# Main Rerun Stage 2 Final Runtime Config

- 当前 Stage 1 final variant：`linear_svm__sigmoid`
  - models: `ridge`, `random_forest`, `xgboost_regressor`
  - spec: `two_stage_v2_core`
- holdout benchmark 固定为：
  - `random_walk_zero`
  - `macro_only_ridge`
  - `macro_only_random_forest`

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "plan_v2.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

MAIN_RERUN_STAGE2_DIR = PROJECT_ROOT / "main_rerun" / "artifacts" / "stage2"
MAIN_RERUN_STAGE2_FINAL_DIR = MAIN_RERUN_STAGE2_DIR / "final_compare"
MAIN_RERUN_STAGE2_FINAL_DIR.mkdir(parents=True, exist_ok=True)

os.environ["STAGE2_HOLDOUT_LABEL"] = "primary_holdout"
os.environ["STAGE2_HOLDOUT_DISPLAY_TITLE"] = "Primary Holdout"
os.environ.setdefault("STAGE2_SEARCH_PROFILE", "smoke")
os.environ["STAGE2_COMPARE_DATA_PATH"] = str(
    MAIN_RERUN_STAGE2_DIR / "stage2_daily_features_long_v2_main_rerun.csv"
)
os.environ["STAGE2_COMPARE_RESULTS_DIR"] = str(MAIN_RERUN_STAGE2_FINAL_DIR)
os.environ["STAGE2_ACTIVE_VARIANTS"] = "linear_svm__sigmoid"
os.environ["STAGE2_ACTIVE_MODELS"] = "ridge,random_forest,xgboost_regressor"
os.environ["STAGE2_SELECTION_MODEL_SPECS"] = "two_stage_v2_core"
os.environ["STAGE2_HOLDOUT_BENCHMARKS"] = "random_walk_zero,macro_only_ridge,macro_only_random_forest"

print("Project root:", PROJECT_ROOT)
print("Main rerun Stage 2 feature dir:", MAIN_RERUN_STAGE2_DIR)
print("Main rerun Stage 2 final dir:", MAIN_RERUN_STAGE2_FINAL_DIR)
print("Stage 1 final variant:", "linear_svm__sigmoid")
print("STAGE2_SEARCH_PROFILE =", os.environ["STAGE2_SEARCH_PROFILE"])

Project root: /Users/horace/Coding/ML finance/project
Main rerun Stage 2 feature dir: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2
Main rerun Stage 2 final dir: /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare
Stage 1 final variant: linear_svm__sigmoid
STAGE2_SEARCH_PROFILE = protocol


In [2]:
from pathlib import Path
import json
import os

os.environ["MPLCONFIGDIR"] = str(Path.cwd() / ".mpl-cache")

import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

DATA_PATH = Path("stage2/artifacts/stage2_daily_features_long_v2.csv")
SEARCH_PROFILE = os.environ.get("STAGE2_SEARCH_PROFILE", "smoke")
RESULTS_DIR = Path(f"stage2/artifacts/nested_cv_model_compare_{SEARCH_PROFILE}_v1")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "gold_ret_1d"
DATE_COL = "date"
TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = os.environ.get("STAGE2_HOLDOUT_LABEL", "primary_holdout")
HOLDOUT_DISPLAY_TITLE = os.environ.get("STAGE2_HOLDOUT_DISPLAY_TITLE", "Primary Hold-out")
DATA_PATH = Path(os.environ.get("STAGE2_COMPARE_DATA_PATH", str(DATA_PATH)))
RESULTS_DIR = Path(os.environ.get("STAGE2_COMPARE_RESULTS_DIR", str(RESULTS_DIR)))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ACTIVE_VARIANTS = [
    key.strip()
    for key in os.environ.get("STAGE2_ACTIVE_VARIANTS", "").split(",")
    if key.strip()
]

MODEL_CANDIDATES = ["ridge", "random_forest", "xgboost_regressor"]
ACTIVE_MODELS = [
    key.strip()
    for key in os.environ.get("STAGE2_ACTIVE_MODELS", ",".join(MODEL_CANDIDATES)).split(",")
    if key.strip()
]
unknown_models = [key for key in ACTIVE_MODELS if key not in MODEL_CANDIDATES]
if unknown_models:
    raise ValueError(f"Unknown STAGE2_ACTIVE_MODELS entries: {unknown_models}")

OUTER_N_SPLITS = 5
OUTER_PURGE_DAYS = 2
OUTER_EMBARGO_DAYS = 5
OUTER_MIN_TRAIN_DAYS = 180
OUTER_MIN_VALID_DAYS = 30

INNER_N_SPLITS = 3
INNER_PURGE_DAYS = 2
INNER_EMBARGO_DAYS = 2
INNER_MIN_TRAIN_DAYS = 120
INNER_MIN_VALID_DAYS = 20

GAP_SPLIT_DAYS = 30
MIN_PREFIX_DAYS_LATER_SEGMENT = 30
RANDOM_STATE = 42

In [3]:
P_TACO_V2_CORE_FEATURES = [
    "p_taco_any",
    "p_taco_top1",
    "p_taco_tail_010",
    "p_taco_any_decay_2d",
]

P_TACO_V2_INTERACTION_FEATURES = [
    "p_taco_any_x_vix_ma_5d_lag1",
    "p_taco_any_x_spx_drawdown_5d_lag1",
]

CORE_CONTROL_FEATURES = [
    "dxy_ret_1d_lag1",
    "real_yield_change_5d_lag1",
    "vix_ma_5d_lag1",
    "vix_change_5d_lag1",
    "spx_ret_1d_lag1",
    "spx_drawdown_5d_lag1",
    "gold_spx_corr_20d_lag1",
]

MODEL_SPEC_CATALOG = {
    "macro_only": CORE_CONTROL_FEATURES,
    "two_stage_v2_core": P_TACO_V2_CORE_FEATURES + CORE_CONTROL_FEATURES,
    "two_stage_v2_interact": P_TACO_V2_CORE_FEATURES + P_TACO_V2_INTERACTION_FEATURES + CORE_CONTROL_FEATURES,
}
SELECTION_MODEL_SPEC_NAMES = [
    key.strip()
    for key in os.environ.get(
        "STAGE2_SELECTION_MODEL_SPECS",
        "two_stage_v2_core",
    ).split(",")
    if key.strip()
]
unknown_selection_model_specs = [key for key in SELECTION_MODEL_SPEC_NAMES if key not in MODEL_SPEC_CATALOG]
if unknown_selection_model_specs:
    raise ValueError(f"Unknown STAGE2_SELECTION_MODEL_SPECS entries: {unknown_selection_model_specs}")
SELECTION_MODEL_SPECS = {key: MODEL_SPEC_CATALOG[key] for key in SELECTION_MODEL_SPEC_NAMES}

HOLDOUT_BENCHMARK_CATALOG = {
    "random_walk_zero": {
        "benchmark_label": "Random Walk (Zero)",
        "kind": "naive_zero",
    },
    "macro_only_ridge": {
        "benchmark_label": "Ridge + macro_only",
        "kind": "model",
        "model_name": "ridge",
        "spec_name": "macro_only",
    },
    "macro_only_random_forest": {
        "benchmark_label": "RandomForest + macro_only",
        "kind": "model",
        "model_name": "random_forest",
        "spec_name": "macro_only",
    },
}
HOLDOUT_BENCHMARK_KEYS = [
    key.strip()
    for key in os.environ.get(
        "STAGE2_HOLDOUT_BENCHMARKS",
        "random_walk_zero,macro_only_ridge,macro_only_random_forest",
    ).split(",")
    if key.strip()
]
unknown_holdout_benchmarks = [key for key in HOLDOUT_BENCHMARK_KEYS if key not in HOLDOUT_BENCHMARK_CATALOG]
if unknown_holdout_benchmarks:
    raise ValueError(f"Unknown STAGE2_HOLDOUT_BENCHMARKS entries: {unknown_holdout_benchmarks}")


def compute_regression_metrics(y_true, y_pred, benchmark_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    benchmark_pred = np.asarray(benchmark_pred, dtype=float)
    benchmark_mse = mean_squared_error(y_true, benchmark_pred)
    if benchmark_mse == 0:
        oos_r2 = np.nan
    else:
        oos_r2 = 1.0 - (mean_squared_error(y_true, y_pred) / benchmark_mse)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "oos_r2": float(oos_r2) if pd.notna(oos_r2) else np.nan,
        "sign_accuracy": float((np.sign(y_pred) == np.sign(y_true)).mean()),
        "actual_mean": float(np.mean(y_true)),
        "pred_mean": float(np.mean(y_pred)),
        "benchmark_mean": float(np.mean(benchmark_pred)),
    }


def split_day_segments(unique_days, gap_days=30):
    unique_days = pd.Index(sorted(pd.Index(unique_days).unique()))
    if len(unique_days) == 0:
        return []

    segments = []
    start = 0
    diffs = np.diff(unique_days.view("int64")) / (24 * 3600 * 1e9)
    for idx, gap in enumerate(diffs, start=1):
        if gap > gap_days:
            segments.append(unique_days[start:idx])
            start = idx
    segments.append(unique_days[start:])
    return segments


def allocate_splits_to_segments(
    segments,
    n_splits,
    min_train_days,
    purge_days,
    min_valid_days,
    embargo_days,
    min_prefix_days_later_segment,
):
    eligible = []
    for seg_idx, segment in enumerate(segments):
        required_days = (
            min_train_days + purge_days + min_valid_days
            if seg_idx == 0
            else min_prefix_days_later_segment + purge_days + min_valid_days
        )
        if len(segment) < required_days:
            continue

        segment_capacity = int(
            (len(segment) - required_days) // max(min_valid_days + embargo_days, 1)
        ) + 1
        if segment_capacity <= 0:
            continue

        eligible.append(
            {
                "segment_idx": seg_idx,
                "segment_len": len(segment),
                "capacity": segment_capacity,
            }
        )

    if not eligible:
        raise ValueError("No eligible day segments available for the requested split design.")

    allocation = {item["segment_idx"]: 0 for item in eligible}
    remaining = n_splits

    for item in eligible:
        if remaining <= 0:
            break
        allocation[item["segment_idx"]] += 1
        remaining -= 1

    while remaining > 0:
        best_choice = None
        for item in eligible:
            current = allocation[item["segment_idx"]]
            if current >= item["capacity"]:
                continue
            score = item["segment_len"] / (current + 1)
            if best_choice is None or score > best_choice[0]:
                best_choice = (score, item["segment_idx"])

        if best_choice is None:
            break

        allocation[best_choice[1]] += 1
        remaining -= 1

    return allocation


def build_segmented_day_splits(
    unique_days,
    n_splits,
    purge_days,
    embargo_days,
    min_train_days,
    min_valid_days,
    gap_days=30,
    min_prefix_days_later_segment=30,
):
    unique_days = pd.Index(sorted(pd.Index(unique_days).unique()))
    segments = split_day_segments(unique_days, gap_days=gap_days)
    allocation = allocate_splits_to_segments(
        segments=segments,
        n_splits=n_splits,
        min_train_days=min_train_days,
        purge_days=purge_days,
        min_valid_days=min_valid_days,
        embargo_days=embargo_days,
        min_prefix_days_later_segment=min_prefix_days_later_segment,
    )

    splits = []
    prior_days = []
    fold_id = 1

    for seg_idx, segment in enumerate(segments):
        n_segment_splits = allocation.get(seg_idx, 0)
        if n_segment_splits == 0:
            prior_days.extend(list(segment))
            continue

        prefix_days = min_train_days if seg_idx == 0 else min_prefix_days_later_segment
        valid_start_min = prefix_days + purge_days
        available_end = len(segment) - min_valid_days
        if available_end < valid_start_min:
            prior_days.extend(list(segment))
            continue

        raw_starts = (
            np.array([valid_start_min])
            if n_segment_splits == 1
            else np.linspace(valid_start_min, available_end, n_segment_splits, dtype=int)
        )
        raw_starts = np.maximum.accumulate(raw_starts)
        previous_valid_end = None

        for local_idx, start_pos in enumerate(raw_starts, start=1):
            if previous_valid_end is not None:
                start_pos = max(start_pos, previous_valid_end + 1 + embargo_days)

            remaining_after = n_segment_splits - local_idx
            max_end_pos = (
                len(segment) - 1
                if remaining_after == 0
                else len(segment) - remaining_after * (min_valid_days + embargo_days) - 1
            )
            next_start = raw_starts[local_idx] if local_idx < n_segment_splits else len(segment)
            end_pos = max(
                start_pos + min_valid_days - 1,
                next_start - 1 - embargo_days,
            )
            end_pos = min(end_pos, max_end_pos)

            train_seg_end = start_pos - purge_days - 1
            train_days = pd.Index(list(prior_days) + list(segment[: train_seg_end + 1]))
            valid_days = segment[start_pos : end_pos + 1]

            purge_start = segment[train_seg_end + 1] if train_seg_end + 1 < len(segment) else pd.NaT
            purge_end = segment[start_pos - 1] if start_pos - 1 < len(segment) else pd.NaT
            embargo_start = segment[end_pos + 1] if end_pos + 1 < len(segment) else pd.NaT
            embargo_end_pos = min(end_pos + embargo_days, len(segment) - 1)
            embargo_end = segment[embargo_end_pos] if end_pos + 1 < len(segment) else pd.NaT

            splits.append(
                {
                    "fold_id": fold_id,
                    "segment_idx": seg_idx,
                    "train_days": train_days,
                    "valid_days": valid_days,
                    "train_start_day": train_days.min(),
                    "train_end_day": train_days.max(),
                    "valid_start_day": valid_days.min(),
                    "valid_end_day": valid_days.max(),
                    "purge_start_day": purge_start,
                    "purge_end_day": purge_end,
                    "embargo_start_day": embargo_start,
                    "embargo_end_day": embargo_end,
                    "train_day_count": len(train_days),
                    "valid_day_count": len(valid_days),
                }
            )
            previous_valid_end = end_pos
            fold_id += 1

        prior_days.extend(list(segment))

    return splits


def make_cv_index_pairs(frame, split_records):
    frame = frame.sort_values(DATE_COL).reset_index(drop=True)
    index_pairs = []
    split_meta_rows = []
    for split in split_records:
        train_idx = frame.index[frame[DATE_COL].isin(split["train_days"])].to_numpy()
        valid_idx = frame.index[frame[DATE_COL].isin(split["valid_days"])].to_numpy()
        if len(train_idx) == 0 or len(valid_idx) == 0:
            continue

        index_pairs.append((train_idx, valid_idx))
        split_meta_rows.append(
            {
                "fold_id": split["fold_id"],
                "segment_idx": split["segment_idx"],
                "train_start_day": split["train_start_day"],
                "train_end_day": split["train_end_day"],
                "valid_start_day": split["valid_start_day"],
                "valid_end_day": split["valid_end_day"],
                "purge_start_day": split["purge_start_day"],
                "purge_end_day": split["purge_end_day"],
                "embargo_start_day": split["embargo_start_day"],
                "embargo_end_day": split["embargo_end_day"],
                "train_day_count": split["train_day_count"],
                "valid_day_count": split["valid_day_count"],
                "train_rows": len(train_idx),
                "valid_rows": len(valid_idx),
            }
        )
    return index_pairs, pd.DataFrame(split_meta_rows)


def choose_inner_cv(frame):
    unique_days = pd.Index(sorted(frame[DATE_COL].dropna().unique()))
    try:
        split_records = build_segmented_day_splits(
            unique_days=unique_days,
            n_splits=INNER_N_SPLITS,
            purge_days=INNER_PURGE_DAYS,
            embargo_days=INNER_EMBARGO_DAYS,
            min_train_days=INNER_MIN_TRAIN_DAYS,
            min_valid_days=INNER_MIN_VALID_DAYS,
            gap_days=GAP_SPLIT_DAYS,
            min_prefix_days_later_segment=MIN_PREFIX_DAYS_LATER_SEGMENT,
        )
        cv_pairs, split_df = make_cv_index_pairs(frame, split_records)
        if len(cv_pairs) >= 2:
            return cv_pairs, split_df, "segmented_purged"
    except ValueError:
        pass

    fallback_splits = min(INNER_N_SPLITS, max(frame[DATE_COL].nunique() - 1, 1))
    if fallback_splits < 2:
        raise ValueError("Not enough unique dates to build inner CV.")

    fallback = TimeSeriesSplit(n_splits=fallback_splits)
    fallback_pairs = list(fallback.split(frame))
    fallback_rows = []
    for idx, (train_idx, valid_idx) in enumerate(fallback_pairs, start=1):
        fallback_rows.append(
            {
                "fold_id": idx,
                "segment_idx": -1,
                "train_start_day": frame.iloc[train_idx][DATE_COL].min(),
                "train_end_day": frame.iloc[train_idx][DATE_COL].max(),
                "valid_start_day": frame.iloc[valid_idx][DATE_COL].min(),
                "valid_end_day": frame.iloc[valid_idx][DATE_COL].max(),
                "purge_start_day": pd.NaT,
                "purge_end_day": pd.NaT,
                "embargo_start_day": pd.NaT,
                "embargo_end_day": pd.NaT,
                "train_day_count": frame.iloc[train_idx][DATE_COL].nunique(),
                "valid_day_count": frame.iloc[valid_idx][DATE_COL].nunique(),
                "train_rows": len(train_idx),
                "valid_rows": len(valid_idx),
            }
        )
    return fallback_pairs, pd.DataFrame(fallback_rows), "time_series_fallback"


def build_model_bundle(model_name, search_profile):
    if model_name == "ridge":
        estimator = Ridge()
        if search_profile == "protocol":
            grid = {"model__alpha": [0.1, 1.0, 10.0]}
        elif search_profile == "targeted_followup":
            grid = {"model__alpha": [10.0, 30.0, 100.0, 300.0]}
        else:
            grid = {"model__alpha": [0.1, 1.0, 10.0, 100.0]}
    elif model_name == "random_forest":
        estimator = RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
        if search_profile == "protocol":
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [3, 5],
                "model__min_samples_leaf": [1, 5],
                "model__max_features": ["sqrt", 0.5],
            }
        elif search_profile == "targeted_followup":
            grid = {
                "model__n_estimators": [200, 500],
                "model__max_depth": [2, 3, 5],
                "model__min_samples_leaf": [1, 5],
                "model__max_features": ["sqrt", 0.5],
            }
        else:
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [3, 5],
                "model__min_samples_leaf": [1, 5],
                "model__max_features": ["sqrt", 0.5],
            }
    elif model_name == "xgboost_regressor":
        estimator = XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
            verbosity=0,
        )
        if search_profile == "protocol":
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [2, 3],
                "model__learning_rate": [0.03, 0.08],
                "model__min_child_weight": [1, 5],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
            }
        elif search_profile == "targeted_followup":
            grid = {
                "model__n_estimators": [200, 500],
                "model__max_depth": [1, 2, 3],
                "model__learning_rate": [0.01, 0.03],
                "model__min_child_weight": [5, 10],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
            }
        else:
            grid = {
                "model__n_estimators": [200],
                "model__max_depth": [2, 3],
                "model__learning_rate": [0.03, 0.08],
                "model__min_child_weight": [1, 5],
                "model__subsample": [0.8],
                "model__colsample_bytree": [0.8],
            }
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return estimator, grid


def make_model_pipeline(features, model_name):
    estimator, grid = build_model_bundle(model_name=model_name, search_profile=SEARCH_PROFILE)
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                features,
            ),
        ],
        remainder="drop",
    )
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )
    return pipeline, grid


def extract_model_importance(estimator, features, variant_key, spec_name, model_name):
    fitted_model = estimator.named_steps["model"]
    if hasattr(fitted_model, "coef_"):
        values = fitted_model.coef_
        importance_type = "coefficient"
    elif hasattr(fitted_model, "feature_importances_"):
        values = fitted_model.feature_importances_
        importance_type = "feature_importance"
    else:
        values = np.repeat(np.nan, len(features))
        importance_type = "not_available"

    return pd.DataFrame(
        {
            "variant_key": variant_key,
            "model_name": model_name,
            "spec_name": spec_name,
            "feature_name": features,
            "importance_value": values,
            "abs_importance_value": np.abs(values),
            "importance_type": importance_type,
        }
    )


def flatten_summary_columns(frame):
    flat_cols = []
    for col in frame.columns:
        if isinstance(col, tuple):
            flat_cols.append("_".join([str(part) for part in col if part]).strip("_"))
        else:
            flat_cols.append(str(col))
    out = frame.copy()
    out.columns = flat_cols
    return out.reset_index()

In [4]:
df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL]).dt.normalize()
df = df.sort_values(["variant_key", DATE_COL]).reset_index(drop=True)

if not ACTIVE_VARIANTS:
    ACTIVE_VARIANTS = sorted(df["variant_key"].dropna().unique().tolist())
missing_variants = [key for key in ACTIVE_VARIANTS if key not in df["variant_key"].dropna().unique()]
if missing_variants:
    raise ValueError(f"Missing active variants in compare input: {missing_variants}")

active_feature_columns = sorted(
    set(
        feature
        for feature_cols in list(SELECTION_MODEL_SPECS.values()) + [MODEL_SPEC_CATALOG["macro_only"]]
        for feature in feature_cols
    )
)
required_columns = sorted(
    set([TARGET_COL, DATE_COL, "stage2_split", "variant_key"] + active_feature_columns)
)
missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

main_df = df[df["stage2_split"].isin([TRAIN_LABEL, HOLDOUT_LABEL])].copy()
main_df = main_df.dropna(subset=[TARGET_COL]).sort_values(["variant_key", DATE_COL]).reset_index(drop=True)

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")
print(f"Main sample rows with non-null target: {len(main_df):,}")
print("Search profile:", SEARCH_PROFILE)

Loaded 2,293 rows and 52 columns from /Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/stage2_daily_features_long_v2_main_rerun.csv
Main sample rows with non-null target: 1,204
Search profile: protocol


In [5]:
split_summary = (
    main_df.groupby(["variant_key", "stage2_split"])
    .agg(
        n_rows=(DATE_COL, "size"),
        n_unique_dates=(DATE_COL, "nunique"),
        first_date=(DATE_COL, "min"),
        last_date=(DATE_COL, "max"),
        target_mean=(TARGET_COL, "mean"),
        target_std=(TARGET_COL, "std"),
    )
    .reset_index()
    .sort_values(["variant_key", "stage2_split"])
)
display(split_summary)

missing_summary = (
    main_df[CORE_CONTROL_FEATURES + P_TACO_V2_CORE_FEATURES + P_TACO_V2_INTERACTION_FEATURES + [TARGET_COL]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .to_frame()
)
display(missing_summary)

,variant_key,stage2_split,n_rows,n_unique_dates,first_date,last_date,target_mean,target_std
0,linear_svm__sigmoid,primary_holdout,74,74,2025-07-15,2025-10-24,0.002761,0.014441
1,linear_svm__sigmoid,train_pool,1130,1130,2017-01-20,2025-07-14,0.000570,0.008912


,missing_rate
dxy_ret_1d_lag1,0.038206
spx_ret_1d_lag1,0.018272
spx_drawdown_5d_lag1,0.018272
vix_change_5d_lag1,0.008306
real_yield_change_5d_lag1,0.000000
vix_ma_5d_lag1,0.000000
gold_spx_corr_20d_lag1,0.000000
p_taco_any,0.000000
p_taco_top1,0.000000
p_taco_tail_010,0.000000


In [6]:
outer_unique_days = pd.Index(
    sorted(main_df.loc[main_df["stage2_split"] == TRAIN_LABEL, DATE_COL].dropna().unique())
)
outer_split_records = build_segmented_day_splits(
    unique_days=outer_unique_days,
    n_splits=OUTER_N_SPLITS,
    purge_days=OUTER_PURGE_DAYS,
    embargo_days=OUTER_EMBARGO_DAYS,
    min_train_days=OUTER_MIN_TRAIN_DAYS,
    min_valid_days=OUTER_MIN_VALID_DAYS,
    gap_days=GAP_SPLIT_DAYS,
    min_prefix_days_later_segment=MIN_PREFIX_DAYS_LATER_SEGMENT,
)
_, outer_splits_df = make_cv_index_pairs(
    frame=main_df[main_df["variant_key"] == ACTIVE_VARIANTS[0]].copy(),
    split_records=outer_split_records,
)
display(outer_splits_df)

,fold_id,segment_idx,train_start_day,train_end_day,valid_start_day,valid_end_day,purge_start_day,purge_end_day,embargo_start_day,embargo_end_day,train_day_count,valid_day_count,train_rows,valid_rows
0,1,0,2017-01-20,2017-10-04,2017-10-09,2018-10-16,2017-10-05,2017-10-06,2018-10-17,2018-10-23,180,259,180,259
1,2,0,2017-01-20,2018-10-19,2018-10-24,2019-11-01,2018-10-22,2018-10-23,2019-11-04,2019-11-08,444,259,444,259
2,3,0,2017-01-20,2019-11-06,2019-11-11,2020-11-17,2019-11-07,2019-11-08,2020-11-18,2020-11-24,708,259,708,259
3,4,0,2017-01-20,2020-11-20,2020-11-25,2021-01-08,2020-11-23,2020-11-24,NaT,NaT,972,30,972,30
4,5,1,2017-01-20,2025-02-28,2025-03-05,2025-07-07,2025-03-03,2025-03-04,2025-07-08,2025-07-14,1034,89,1034,89


In [7]:
fig, ax = plt.subplots(figsize=(16, 6))
for _, row in outer_splits_df.iterrows():
    ax.plot([row["train_start_day"], row["train_end_day"]], [row["fold_id"], row["fold_id"]], color="#457b9d", linewidth=6, solid_capstyle="butt")
    ax.plot([row["valid_start_day"], row["valid_end_day"]], [row["fold_id"], row["fold_id"]], color="#e76f51", linewidth=6, solid_capstyle="butt")
ax.set_title("Stage 2 Outer Splits: Train (blue) vs Validation (red)")
ax.set_xlabel("date")
ax.set_ylabel("fold_id")
plt.tight_layout()
plt.show()

In [8]:
outer_fold_metric_rows = []
outer_prediction_frames = []
best_param_rows = []
holdout_metric_rows = []
holdout_prediction_frames = []
holdout_importance_frames = []
holdout_benchmark_rows = []
holdout_benchmark_prediction_frames = []

for variant_key in ACTIVE_VARIANTS:
    variant_df = (
        main_df[main_df["variant_key"] == variant_key]
        .sort_values(DATE_COL)
        .reset_index(drop=True)
    )
    train_pool_df = variant_df[variant_df["stage2_split"] == TRAIN_LABEL].copy().reset_index(drop=True)
    holdout_df = variant_df[variant_df["stage2_split"] == HOLDOUT_LABEL].copy().reset_index(drop=True)

    print(
        f"{variant_key}: train_pool={len(train_pool_df):,} rows / {train_pool_df[DATE_COL].nunique():,} days, "
        f"holdout={len(holdout_df):,} rows / {holdout_df[DATE_COL].nunique():,} days"
    )

    for model_name in ACTIVE_MODELS:
        print(f"  - model {model_name}")

        for spec_name, feature_cols in SELECTION_MODEL_SPECS.items():
            print(f"    * spec {spec_name}")

            for outer_split in outer_split_records:
                outer_train = train_pool_df[train_pool_df[DATE_COL].isin(outer_split["train_days"])].copy().reset_index(drop=True)
                outer_valid = train_pool_df[train_pool_df[DATE_COL].isin(outer_split["valid_days"])].copy().reset_index(drop=True)

                inner_cv_pairs, inner_splits_df, inner_strategy = choose_inner_cv(outer_train)
                pipeline, param_grid = make_model_pipeline(feature_cols, model_name=model_name)
                search = GridSearchCV(
                    estimator=pipeline,
                    param_grid=param_grid,
                    scoring="neg_root_mean_squared_error",
                    cv=inner_cv_pairs,
                    refit=True,
                    n_jobs=None,
                )
                search.fit(outer_train[feature_cols], outer_train[TARGET_COL])

                valid_pred = search.best_estimator_.predict(outer_valid[feature_cols])
                outer_benchmark_pred = np.repeat(
                    outer_train[TARGET_COL].mean(),
                    len(outer_valid),
                )
                fold_metrics = compute_regression_metrics(
                    outer_valid[TARGET_COL].to_numpy(),
                    valid_pred,
                    benchmark_pred=outer_benchmark_pred,
                )
                fold_metrics.update(
                    {
                        "variant_key": variant_key,
                        "model_name": model_name,
                        "spec_name": spec_name,
                        "fold_id": outer_split["fold_id"],
                        "segment_idx": outer_split["segment_idx"],
                        "inner_cv_strategy": inner_strategy,
                        "best_params_json": json.dumps(search.best_params_, ensure_ascii=False),
                        "best_inner_rmse": float(-search.best_score_),
                        "train_start_day": outer_split["train_start_day"],
                        "train_end_day": outer_split["train_end_day"],
                        "valid_start_day": outer_split["valid_start_day"],
                        "valid_end_day": outer_split["valid_end_day"],
                        "train_day_count": outer_split["train_day_count"],
                        "valid_day_count": outer_split["valid_day_count"],
                        "train_rows": len(outer_train),
                        "valid_rows": len(outer_valid),
                    }
                )
                outer_fold_metric_rows.append(fold_metrics)

                outer_prediction_frames.append(
                    pd.DataFrame(
                        {
                            "date": outer_valid[DATE_COL].to_numpy(),
                            "variant_key": variant_key,
                            "model_name": model_name,
                            "spec_name": spec_name,
                            "fold_id": outer_split["fold_id"],
                            "y_true": outer_valid[TARGET_COL].to_numpy(),
                            "y_pred": valid_pred,
                        }
                    )
                )

                best_param_rows.append(
                    {
                        "variant_key": variant_key,
                        "model_name": model_name,
                        "spec_name": spec_name,
                        "fold_id": outer_split["fold_id"],
                        "segment_idx": outer_split["segment_idx"],
                        "best_params_json": json.dumps(search.best_params_, ensure_ascii=False),
                        "best_inner_rmse": float(-search.best_score_),
                        "inner_cv_strategy": inner_strategy,
                        "feature_list_json": json.dumps(feature_cols, ensure_ascii=False),
                    }
                )

            full_inner_pairs, full_inner_splits_df, full_inner_strategy = choose_inner_cv(train_pool_df)
            full_pipeline, full_param_grid = make_model_pipeline(feature_cols, model_name=model_name)
            full_search = GridSearchCV(
                estimator=full_pipeline,
                param_grid=full_param_grid,
                scoring="neg_root_mean_squared_error",
                cv=full_inner_pairs,
                refit=True,
                n_jobs=None,
            )
            full_search.fit(train_pool_df[feature_cols], train_pool_df[TARGET_COL])
            holdout_pred = full_search.best_estimator_.predict(holdout_df[feature_cols])
            holdout_benchmark_pred = np.repeat(
                train_pool_df[TARGET_COL].mean(),
                len(holdout_df),
            )
            holdout_metrics = compute_regression_metrics(
                holdout_df[TARGET_COL].to_numpy(),
                holdout_pred,
                benchmark_pred=holdout_benchmark_pred,
            )
            holdout_metrics.update(
                {
                    "variant_key": variant_key,
                    "model_name": model_name,
                    "spec_name": spec_name,
                    "best_params_json": json.dumps(full_search.best_params_, ensure_ascii=False),
                    "best_inner_rmse": float(-full_search.best_score_),
                    "inner_cv_strategy": full_inner_strategy,
                    "n_obs": len(holdout_df),
                }
            )
            holdout_metric_rows.append(holdout_metrics)

            holdout_prediction_frames.append(
                pd.DataFrame(
                    {
                        "date": holdout_df[DATE_COL].to_numpy(),
                        "variant_key": variant_key,
                        "model_name": model_name,
                        "spec_name": spec_name,
                        "y_true": holdout_df[TARGET_COL].to_numpy(),
                        "y_pred": holdout_pred,
                    }
                )
            )

            holdout_importance_frames.append(
                extract_model_importance(
                    estimator=full_search.best_estimator_,
                    features=feature_cols,
                    variant_key=variant_key,
                    spec_name=spec_name,
                    model_name=model_name,
                )
            )

    train_pool_benchmark_pred = np.repeat(
        train_pool_df[TARGET_COL].mean(),
        len(holdout_df),
    )
    for benchmark_key in HOLDOUT_BENCHMARK_KEYS:
        benchmark_config = HOLDOUT_BENCHMARK_CATALOG[benchmark_key]
        if benchmark_config["kind"] == "naive_zero":
            holdout_pred = np.zeros(len(holdout_df))
            holdout_metrics = compute_regression_metrics(
                holdout_df[TARGET_COL].to_numpy(),
                holdout_pred,
                benchmark_pred=train_pool_benchmark_pred,
            )
            holdout_metrics.update(
                {
                    "variant_key": variant_key,
                    "benchmark_key": benchmark_key,
                    "benchmark_label": benchmark_config["benchmark_label"],
                    "model_name": "benchmark",
                    "spec_name": benchmark_key,
                    "reference_model_name": np.nan,
                    "reference_spec_name": np.nan,
                    "best_params_json": "",
                    "best_inner_rmse": np.nan,
                    "inner_cv_strategy": "naive_zero",
                    "n_obs": len(holdout_df),
                }
            )
            holdout_benchmark_rows.append(holdout_metrics)
            holdout_benchmark_prediction_frames.append(
                pd.DataFrame(
                    {
                        "date": holdout_df[DATE_COL].to_numpy(),
                        "variant_key": variant_key,
                        "benchmark_key": benchmark_key,
                        "benchmark_label": benchmark_config["benchmark_label"],
                        "y_true": holdout_df[TARGET_COL].to_numpy(),
                        "y_pred": holdout_pred,
                    }
                )
            )
            continue

        benchmark_model_name = benchmark_config["model_name"]
        benchmark_spec_name = benchmark_config["spec_name"]
        benchmark_feature_cols = MODEL_SPEC_CATALOG[benchmark_spec_name]
        full_inner_pairs, _, full_inner_strategy = choose_inner_cv(train_pool_df)
        full_pipeline, full_param_grid = make_model_pipeline(
            benchmark_feature_cols,
            model_name=benchmark_model_name,
        )
        full_search = GridSearchCV(
            estimator=full_pipeline,
            param_grid=full_param_grid,
            scoring="neg_root_mean_squared_error",
            cv=full_inner_pairs,
            refit=True,
            n_jobs=None,
        )
        full_search.fit(train_pool_df[benchmark_feature_cols], train_pool_df[TARGET_COL])
        holdout_pred = full_search.best_estimator_.predict(holdout_df[benchmark_feature_cols])
        holdout_metrics = compute_regression_metrics(
            holdout_df[TARGET_COL].to_numpy(),
            holdout_pred,
            benchmark_pred=train_pool_benchmark_pred,
        )
        holdout_metrics.update(
            {
                "variant_key": variant_key,
                "benchmark_key": benchmark_key,
                "benchmark_label": benchmark_config["benchmark_label"],
                "model_name": "benchmark",
                "spec_name": benchmark_key,
                "reference_model_name": benchmark_model_name,
                "reference_spec_name": benchmark_spec_name,
                "best_params_json": json.dumps(full_search.best_params_, ensure_ascii=False),
                "best_inner_rmse": float(-full_search.best_score_),
                "inner_cv_strategy": full_inner_strategy,
                "n_obs": len(holdout_df),
            }
        )
        holdout_benchmark_rows.append(holdout_metrics)
        holdout_benchmark_prediction_frames.append(
            pd.DataFrame(
                {
                    "date": holdout_df[DATE_COL].to_numpy(),
                    "variant_key": variant_key,
                    "benchmark_key": benchmark_key,
                    "benchmark_label": benchmark_config["benchmark_label"],
                    "y_true": holdout_df[TARGET_COL].to_numpy(),
                    "y_pred": holdout_pred,
                }
            )
        )

outer_fold_metrics_df = pd.DataFrame(outer_fold_metric_rows)
outer_predictions_df = pd.concat(outer_prediction_frames, ignore_index=True)
best_params_df = pd.DataFrame(best_param_rows)
holdout_metrics_df = pd.DataFrame(holdout_metric_rows).sort_values(["variant_key", "model_name", "spec_name"]).reset_index(drop=True)
holdout_predictions_df = pd.concat(holdout_prediction_frames, ignore_index=True)
holdout_importances_df = pd.concat(holdout_importance_frames, ignore_index=True)
holdout_benchmark_metrics_df = pd.DataFrame(holdout_benchmark_rows).sort_values(["variant_key", "benchmark_key"]).reset_index(drop=True)
holdout_benchmark_predictions_df = pd.concat(holdout_benchmark_prediction_frames, ignore_index=True)

outer_summary_df = flatten_summary_columns(
    outer_fold_metrics_df.groupby(["variant_key", "model_name", "spec_name"])[["rmse", "mae", "r2", "oos_r2", "sign_accuracy"]]
    .agg(["mean", "std"])
)

selected_model_rows = []
for variant_key in ACTIVE_VARIANTS:
    variant_outer = outer_summary_df[outer_summary_df["variant_key"] == variant_key].copy()
    selected_row = (
        variant_outer.sort_values(
            ["rmse_mean", "mae_mean", "oos_r2_mean"],
            ascending=[True, True, False],
        )
        .iloc[0]
        .to_dict()
    )
    selected_model_rows.append(selected_row)
selected_model_df = pd.DataFrame(selected_model_rows)

display(outer_summary_df.sort_values(["variant_key", "model_name", "rmse_mean"]))
display(selected_model_df[["variant_key", "model_name", "spec_name", "rmse_mean", "mae_mean", "oos_r2_mean", "sign_accuracy_mean"]])
display(holdout_metrics_df[["variant_key", "model_name", "spec_name", "rmse", "mae", "r2", "oos_r2", "sign_accuracy"]])
display(holdout_benchmark_metrics_df[["variant_key", "benchmark_label", "rmse", "mae", "r2", "oos_r2", "sign_accuracy"]])

linear_svm__sigmoid: train_pool=1,130 rows / 1,130 days, holdout=74 rows / 74 days
  - model ridge
    * spec two_stage_v2_core


  - model random_forest
    * spec two_stage_v2_core


  - model xgboost_regressor
    * spec two_stage_v2_core


,variant_key,model_name,spec_name,rmse_mean,rmse_std,mae_mean,mae_std,r2_mean,r2_std,oos_r2_mean,oos_r2_std,sign_accuracy_mean,sign_accuracy_std
0,linear_svm__sigmoid,random_forest,two_stage_v2_core,0.009845,0.002732,0.007236,0.002011,-0.019075,0.055373,-0.013328,0.054852,0.513932,0.064048
1,linear_svm__sigmoid,ridge,two_stage_v2_core,0.010156,0.002723,0.007396,0.002009,-0.097848,0.151841,-0.091613,0.150513,0.506495,0.037496
2,linear_svm__sigmoid,xgboost_regressor,two_stage_v2_core,0.010001,0.002650,0.007394,0.001969,-0.060304,0.093164,-0.054305,0.092217,0.506488,0.057522


,variant_key,model_name,spec_name,rmse_mean,mae_mean,oos_r2_mean,sign_accuracy_mean
0,linear_svm__sigmoid,random_forest,two_stage_v2_core,0.009845,0.007236,-0.013328,0.513932


,variant_key,model_name,spec_name,rmse,mae,r2,oos_r2,sign_accuracy
0,linear_svm__sigmoid,random_forest,two_stage_v2_core,0.014453,0.009367,-0.015483,0.007664,0.648649
1,linear_svm__sigmoid,ridge,two_stage_v2_core,0.014572,0.009415,-0.032234,-0.008706,0.527027
2,linear_svm__sigmoid,xgboost_regressor,two_stage_v2_core,0.014511,0.009466,-0.023555,-0.000224,0.581081


,variant_key,benchmark_label,rmse,mae,r2,oos_r2,sign_accuracy
0,linear_svm__sigmoid,RandomForest + macro_only,0.014471,0.009331,-0.017907,0.005295,0.635135
1,linear_svm__sigmoid,Ridge + macro_only,0.014610,0.009379,-0.037613,-0.013962,0.581081
2,linear_svm__sigmoid,Random Walk (Zero),0.014606,0.009495,-0.037045,-0.013407,0.000000


In [9]:
holdout_compare_rows = []
for _, selected_row in selected_model_df.iterrows():
    variant_key = selected_row["variant_key"]
    selected_holdout_row = holdout_metrics_df[
        (holdout_metrics_df["variant_key"] == variant_key)
        & (holdout_metrics_df["model_name"] == selected_row["model_name"])
        & (holdout_metrics_df["spec_name"] == selected_row["spec_name"])
    ].iloc[0]

    holdout_compare_rows.append(
        {
            "variant_key": variant_key,
            "benchmark_key": "selected_final_model",
            "benchmark_label": f'{selected_row["model_name"]} + {selected_row["spec_name"]}',
            "rmse": float(selected_holdout_row["rmse"]),
            "mae": float(selected_holdout_row["mae"]),
            "r2": float(selected_holdout_row["r2"]),
            "oos_r2": float(selected_holdout_row["oos_r2"]),
            "sign_accuracy": float(selected_holdout_row["sign_accuracy"]),
            "delta_rmse_vs_selected": 0.0,
            "delta_mae_vs_selected": 0.0,
            "delta_r2_vs_selected": 0.0,
            "delta_oos_r2_vs_selected": 0.0,
            "delta_sign_acc_vs_selected": 0.0,
        }
    )

    for _, benchmark_row in holdout_benchmark_metrics_df[
        holdout_benchmark_metrics_df["variant_key"] == variant_key
    ].iterrows():
        holdout_compare_rows.append(
            {
                "variant_key": variant_key,
                "benchmark_key": benchmark_row["benchmark_key"],
                "benchmark_label": benchmark_row["benchmark_label"],
                "rmse": float(benchmark_row["rmse"]),
                "mae": float(benchmark_row["mae"]),
                "r2": float(benchmark_row["r2"]),
                "oos_r2": float(benchmark_row["oos_r2"]),
                "sign_accuracy": float(benchmark_row["sign_accuracy"]),
                "delta_rmse_vs_selected": float(benchmark_row["rmse"] - selected_holdout_row["rmse"]),
                "delta_mae_vs_selected": float(benchmark_row["mae"] - selected_holdout_row["mae"]),
                "delta_r2_vs_selected": float(benchmark_row["r2"] - selected_holdout_row["r2"]),
                "delta_oos_r2_vs_selected": float(benchmark_row["oos_r2"] - selected_holdout_row["oos_r2"]),
                "delta_sign_acc_vs_selected": float(benchmark_row["sign_accuracy"] - selected_holdout_row["sign_accuracy"]),
            }
        )
holdout_compare = pd.DataFrame(holdout_compare_rows).sort_values(["variant_key", "delta_rmse_vs_selected"])
display(holdout_compare)

,variant_key,benchmark_key,benchmark_label,rmse,mae,r2,oos_r2,sign_accuracy,delta_rmse_vs_selected,delta_mae_vs_selected,delta_r2_vs_selected,delta_oos_r2_vs_selected,delta_sign_acc_vs_selected
0,linear_svm__sigmoid,selected_final_model,random_forest + two_stage_v2_core,0.014453,0.009367,-0.015483,0.007664,0.648649,0.000000,0.000000,0.000000,0.000000,0.000000
1,linear_svm__sigmoid,macro_only_random_forest,RandomForest + macro_only,0.014471,0.009331,-0.017907,0.005295,0.635135,0.000017,-0.000035,-0.002423,-0.002368,-0.013514
3,linear_svm__sigmoid,random_walk_zero,Random Walk (Zero),0.014606,0.009495,-0.037045,-0.013407,0.000000,0.000153,0.000128,-0.021562,-0.021070,-0.648649
2,linear_svm__sigmoid,macro_only_ridge,Ridge + macro_only,0.014610,0.009379,-0.037613,-0.013962,0.581081,0.000157,0.000012,-0.022130,-0.021626,-0.067568


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(
    data=outer_summary_df,
    x="model_name",
    y="rmse_mean",
    ax=axes[0],
    palette="deep",
)
axes[0].set_title("Outer CV RMSE by Model")
axes[0].set_xlabel("model_name")
axes[0].set_ylabel("rmse_mean")

sns.barplot(
    data=holdout_compare,
    x="benchmark_label",
    y="rmse",
    ax=axes[1],
    palette="muted",
)
axes[1].set_title(f"{HOLDOUT_DISPLAY_TITLE} RMSE: Final Model vs Benchmarks")
axes[1].set_xlabel("benchmark_label")
axes[1].set_ylabel("rmse")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [11]:
takeaway_rows = []
for _, selected_row in selected_model_df.iterrows():
    variant_key = selected_row["variant_key"]
    variant_compare = holdout_compare[holdout_compare["variant_key"] == variant_key].copy()
    best_benchmark = (
        variant_compare[variant_compare["benchmark_key"] != "selected_final_model"]
        .sort_values("delta_rmse_vs_selected")
        .iloc[0]
    )
    takeaway_rows.append(
        {
            "variant_key": variant_key,
            "selected_model_name": selected_row["model_name"],
            "selected_spec_name": selected_row["spec_name"],
            "outer_rmse_mean": float(selected_row["rmse_mean"]),
            "best_benchmark_label": best_benchmark["benchmark_label"],
            "holdout_delta_rmse_best_benchmark_vs_selected": float(best_benchmark["delta_rmse_vs_selected"]),
            "holdout_delta_oos_r2_best_benchmark_vs_selected": float(best_benchmark["delta_oos_r2_vs_selected"]),
        }
    )
takeaway_df = pd.DataFrame(takeaway_rows).sort_values(["variant_key", "holdout_delta_rmse_best_benchmark_vs_selected"])
display(takeaway_df)

,variant_key,selected_model_name,selected_spec_name,outer_rmse_mean,best_benchmark_label,holdout_delta_rmse_best_benchmark_vs_selected,holdout_delta_oos_r2_best_benchmark_vs_selected
0,linear_svm__sigmoid,random_forest,two_stage_v2_core,0.009845,RandomForest + macro_only,0.000017,-0.002368


In [12]:
outer_splits_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_splits.csv", index=False)
best_params_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_best_params_by_fold.csv", index=False)
outer_fold_metrics_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_fold_metrics.csv", index=False)
outer_summary_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_summary_metrics.csv", index=False)
outer_predictions_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_outer_predictions.csv", index=False)
holdout_metrics_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_metrics.csv", index=False)
holdout_predictions_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_predictions.csv", index=False)
holdout_importances_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_importances.csv", index=False)
selected_model_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_selected_model.csv", index=False)
holdout_benchmark_metrics_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_benchmark_metrics.csv", index=False)
holdout_benchmark_predictions_df.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_benchmark_predictions.csv", index=False)
holdout_compare.to_csv(RESULTS_DIR / "stage2_nested_cv_model_compare_holdout_compare.csv", index=False)

print("Saved model-compare artifacts:")
for path in sorted(RESULTS_DIR.glob("stage2_nested_cv_model_compare_*.csv")):
    print(path)

Saved model-compare artifacts:
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare/stage2_nested_cv_model_compare_best_params_by_fold.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare/stage2_nested_cv_model_compare_holdout_benchmark_metrics.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare/stage2_nested_cv_model_compare_holdout_benchmark_predictions.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare/stage2_nested_cv_model_compare_holdout_compare.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare/stage2_nested_cv_model_compare_holdout_importances.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare/stage2_nested_cv_model_compare_holdout_metrics.csv
/Users/horace/Coding/ML finance/project/main_rerun/artifacts/stage2/final_compare/stage2_nested_cv_model_compare_holdout_predictions.csv
/U